# Chạy thử TADiSR CogView4 x4 trên Kaggle

Gắn các Dataset **packages**, **models** và **PDF/ảnh nguồn** đã có, bật GPU, đặt **Internet Off** rồi chọn **Run All**. Cấu hình mặc định dùng một GPU hỗ trợ BF16 và đủ VRAM, ưu tiên **1 × L4**. Chọn `PRECISION = "fp32_sharded"` để chạy chế độ FP32 chia nhiều GPU. Notebook dùng manifest wheel TADiSR của dự án và Torch/CUDA có sẵn của Kaggle. Bundle wheel hiện có nhắm tới Python 3.13.

Mặc định chạy trang PDF 54 của sách `4.pdf`, giới hạn cạnh dài đầu vào ở 256 px để thử một trang với ít tile. Điền `INPUT_FILE` để đổi sang PDF hoặc ảnh. Đặt `MAX_INPUT_EDGE = None` để giữ kích thước ảnh đã đọc/render ở `PDF_DPI`. `TILE` và `OVERLAP` tính trên ảnh đã phóng 4×. Notebook thử tile 384 trước và tự giảm tile nếu GPU hết VRAM trong lúc inference. Mỗi lần đổi tile chạy lại trang với cùng seed trên model đang nạp sẵn.

Mỗi lượt ghi `original.png` (đầu vào thật của model), `tadisr_x4.png`, `text_mask.png`, `run.json` và `run.log` dưới `/kaggle/working/tadisr_smoke/`. PNG đầu vào có thể đã thu nhỏ. Text mask là đầu ra chẩn đoán của TADiSR. Nếu thiếu prompt cache, notebook nạp text encoder ở BF16 trên GPU khi đủ chỗ, hoặc dùng CPU BF16. Text encoder chạy ở tiến trình riêng và nhả tài nguyên trước khi nạp model phục dựng.

Mặc định VAE và transformer được nạp BF16 thẳng vào GPU, với các shard của transformer được đọc song song. `PARALLEL_SHARD_LOADING = False` chuyển về đọc tuần tự khi cần kiểm tra I/O hoặc RAM. Chọn `STAGE_WEIGHTS = True` nếu cần thử đọc qua ổ tạm do Dataset chậm. Log báo trạng thái tiến trình mỗi 30 giây khi im lặng. `run.json` ghi dtype, GPU, đỉnh VRAM, kích thước tile thực dùng và thời gian từng bước.

Prompt đã tạo được lưu thêm dưới `tadisr_smoke/prompts/`. Để dùng ở phiên mới, tải file này về, gắn lại qua Kaggle Dataset và điền `PROMPT_FILE` trong ô cấu hình.

In [ ]:
# Điền đường dẫn khi có nhiều Dataset khớp. Để trống để tự tìm.
PACKAGES_DIR = ''
MODELS_DIR = ''
INPUT_FILE = '/kaggle/input/datasets/mduy2911/nlp-yhct/4.pdf'
PDF_PAGE = 54   # Số thứ tự trang trong PDF, đếm từ 1.
PDF_DPI = 96
MAX_INPUT_EDGE = 256

PROMPT_FILE = ''  # Tùy chọn: prompt đã tạo từ cùng bộ CogView4, dưới /kaggle/input.
PRECISION = 'bf16'  # hoặc 'fp32_sharded' để đối chiếu.
PROMPT_DEVICE = 'auto'  # 'auto' dùng GPU nếu đủ VRAM, nếu thiếu thì dùng CPU BF16.
PARALLEL_SHARD_LOADING = True
STAGE_WEIGHTS = False
STAGE_TIMEOUT_MINUTES = 45
PROMPT_TIMEOUT_MINUTES = 75
LOAD_TIMEOUT_MINUTES = 45

GPU_IDS = None  # BF16 dùng GPU thấy đầu tiên; FP32 sharded dùng các GPU được cấp.
MAX_GPU_MEMORY = '18GiB'
TILE = 384  # Kích thước thử đầu tiên; có thể giảm tự động khi CUDA OOM.
OVERLAP = 32
SEED = 42

In [ ]:
import hashlib
import importlib.util
import json
import os
import queue
import shutil
import threading
import time
import subprocess
import sys
import tempfile
from pathlib import Path

from packaging.tags import sys_tags
from packaging.utils import parse_wheel_filename

INPUT_ROOT = Path('/kaggle/input')
OUTPUT_ROOT = Path('/kaggle/working/tadisr_smoke')
RUNTIME = Path('/kaggle/working/.tadisr_smoke')
PLAN_PATH = Path('image_restoration/manifests/tadisr_runtime.json')
ASSET_PATH = Path('image_restoration/tadisr/assets.json')
ADAPTER_NAME = 'tadisr_cogview4_realce_17500.pkl'
ADAPTER_SHA256 = '7cec8d0917305037cbafcb1b1bf81062add1613b561f5d7e2d1e599fe0baf5e0'


def input_path(path):
    path = Path(path).resolve()
    if not path.is_relative_to(INPUT_ROOT.resolve()):
        raise ValueError(f'Chọn đường dẫn trong {INPUT_ROOT}: {path}')
    return path


def bundle_file(root, relative):
    root = root.resolve()
    path = (root / relative).resolve()
    if not path.is_relative_to(root):
        raise ValueError(f'Đường dẫn ngoài bundle: {relative}')
    return path


def find_bundle(configured, marker, wrapper, setting):
    if configured:
        root = input_path(configured)
        candidates = [root / marker, root / wrapper / marker]
    else:
        candidates = [p for p in INPUT_ROOT.rglob(marker.name)
                      if p.parts[-len(marker.parts):] == marker.parts]
    matches = sorted({p for p in candidates if p.is_file()})
    if len(matches) != 1:
        raise ValueError(f'Gắn Dataset và điền {setting}; tìm thấy {len(matches)}: {matches}')
    return input_path(matches[0]).parents[len(marker.parts) - 1]


def find_input(configured):
    if configured:
        path = input_path(configured)
        if not path.is_file():
            raise FileNotFoundError(path)
        return path
    matches = sorted(p for p in INPUT_ROOT.rglob('4.pdf') if p.is_file())
    if len(matches) != 1:
        raise ValueError(f'Điền INPUT_FILE; tìm thấy {len(matches)} bản 4.pdf: {matches}')
    return input_path(matches[0])


def sha256(path):
    with path.open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()


def read_json(path):
    return json.loads(path.read_text(encoding='utf8'))


def verify_assets(folder, prompt_override=None):
    base = folder / 'CogView4'
    manifest = folder / 'assets.json'
    indexed = {entry['path']: entry for entry in read_json(manifest)['files']}
    prompt = prompt_override or base / 'saved_prompt_tokens_nocfg.pt'
    required = [ADAPTER_NAME]
    required.extend('CogView4/' + part for part in (
        'model_index.json', 'scheduler/scheduler_config.json', 'transformer/config.json',
        'vae/config.json', 'vae/diffusion_pytorch_model.safetensors',
    ))
    indexes = ['transformer/diffusion_pytorch_model.safetensors.index.json']
    inference_required = set(required)
    if not prompt.is_file():
        indexes.append('text_encoder/model.safetensors.index.json')
        required.extend('CogView4/' + part for part in (
            'tokenizer/tokenizer.json', 'tokenizer/tokenizer_config.json',
            'tokenizer/special_tokens_map.json', 'text_encoder/config.json',
        ))
    for relative in indexes:
        index = bundle_file(base, relative)
        required.append('CogView4/' + relative)
        for shard in set(read_json(index)['weight_map'].values()):
            required.append((index.parent / shard).relative_to(folder).as_posix())
        if relative.startswith('transformer/'):
            inference_required.update(required_file for required_file in required
                                      if required_file.startswith('CogView4/transformer/'))
    for relative in sorted(required):
        path = bundle_file(folder, relative)
        if not path.is_file():
            raise FileNotFoundError(path)
        if relative not in indexed or indexed[relative]['bytes'] != path.stat().st_size:
            raise ValueError(f'Asset thiếu trong manifest hoặc sai kích thước: {path}')
    adapter = folder / ADAPTER_NAME
    if sha256(adapter) != ADAPTER_SHA256:
        raise ValueError(f'Sai SHA-256 của adapter TADiSR: {adapter}')
    prompt_record = indexed.get('CogView4/saved_prompt_tokens_nocfg.pt')
    if prompt.is_file() and prompt_record:
        if sha256(prompt) != prompt_record['sha256']:
            raise ValueError(f'Sai SHA-256 của prompt cache: {prompt}')
    inference_files = [indexed[name] for name in sorted(inference_required)]
    encoder_bytes = sum(indexed[name]['bytes'] for name in required
                        if name.startswith('CogView4/text_encoder/') and name.endswith('.safetensors'))
    return base, adapter, prompt if prompt.is_file() else None, inference_files, encoder_bytes


def install_overlay(packages, needs_pdf, env):
    manifest = packages / PLAN_PATH
    plan = read_json(manifest)
    root = manifest.parent.parent
    tags = set(sys_tags())
    wheels = []
    for record in plan['wheels']:
        path = bundle_file(root, record['path'])
        if not path.is_file() or sha256(path) != record['sha256']:
            raise ValueError(f'Thiếu wheel offline hoặc sai SHA-256: {path}')
        if not tags.intersection(parse_wheel_filename(path.name)[3]):
            raise RuntimeError(f'Python/Kaggle không tương thích với wheel: {path.name}')
        wheels.append(path)
    if (needs_pdf and importlib.util.find_spec('pypdfium2') is None
            and not any(p.name.startswith('pypdfium2-') for p in wheels)):
        pdf_wheels = sorted(p for p in packages.rglob('pypdfium2-*.whl')
                            if tags.intersection(parse_wheel_filename(p.name)[3]))
        if not pdf_wheels:
            raise FileNotFoundError(f'Thiếu wheel PDFium offline dưới {packages}')
        wheels.append(pdf_wheels[-1])
    key = hashlib.sha256(json.dumps({
        'python': sys.version, 'executable': sys.executable, 'plan': plan,
        'pdfium': [sha256(p) for p in wheels if p.name.startswith('pypdfium2-')],
    }, sort_keys=True).encode()).hexdigest()[:16]
    folder = RUNTIME / f'overlay_{key}'
    python = folder / 'bin/python'
    done = folder / 'install.ok'
    if not done.is_file() or not python.is_file():
        subprocess.run([sys.executable, '-m', 'venv', '--without-pip',
                        '--system-site-packages', str(folder)], env=env, check=True)
        subprocess.run([str(python), '-m', 'pip', '--isolated', 'install', '--no-index',
                        '--ignore-installed', '--no-deps', *map(str, wheels)], env=env, check=True)
        done.touch()
    return python


def prep_worker_environment():
    env = dict(os.environ)
    for key in ('PYTHONPATH', 'PYTHONHOME', 'PYTHONUSERBASE'):
        env.pop(key, None)
    env.update(PYTHONNOUSERSITE='1', PYTHONUTF8='1', HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1', DIFFUSERS_OFFLINE='1', HF_DATASETS_OFFLINE='1', HF_HUB_DISABLE_TELEMETRY='1', PIP_NO_INDEX='1', PIP_DISABLE_PIP_VERSION_CHECK='1', TOKENIZERS_PARALLELISM='false', KERAS_BACKEND='torch')
    return env

### Mã TADiSR đóng gói sẵn

Mã inference và giấy phép từ bản upstream đã ghim trong dự án được nhúng để notebook chạy độc lập với checkout. [TADiSR upstream](https://github.com/mingcv/TADiSR)

In [ ]:
import base64
import zlib

UPSTREAM_REVISION = '9d910f8f67224fae2f57e3d63d68396eebc0b671'
UPSTREAM_SHA256 = 'cf90b8dc38c68231ad4f612b647939d8dc7bde7e90d7570a0e93818b02312d46'
UPSTREAM_B85 = (
    'c-rl~jeFZRk~sXYpzgC*Dv@Z*UrF7n&pEb}`nO5!vz_+(+J0zSqHH!2sgjg!ZJPi6oezKnKv0t1bZ_^r`w@#IFc=I5GlRik@M-1k;oiaf(}R^A'
    'vGVTlOuUWzQIba&$wji4PG(trb6W_n-xBL<>s#W#;&GhZh##jSOa`K#CPfxsPm46mvBGhbjpICz(?rC%xQ(*tdM0kNFe#!zTMV-(66sL%Z^P^+'
    'YKtNTATu$EvK%U;*F_j72s{M1li5WAMT=X2lc&StKFlI&GR*U|ABO-?4ATB|93@3qV1vVW6y?GzZX<EAaw@A_thA_|K@^TI0Jy-<>Z7=ii`#Ts'
    'h%CyX8+`=R7ID%aO$UgF`aFU@%SNy|Q3SdRV5d2B1CecuaXN^H_&1`yO{UkQIKOR+L7X#*;AM_4iMnm<%W9g5JQ@LF03rtDslye1R0y%3AXy7Z'
    's2tzi-=<@;FEL;=oMs6$8d0S|3WTFJe~tPDzQOXtbTmruv3D?bgBUxN@8Hm#!N>4Ay^E+nJnBhW0J@9?4#cDyUim4%4M!t!9Z9M|%Ru8$?^1>s'
    '<OQ%m9FD{!&8SJct3ILd<AFFmdVTiu?#Y2TJQc?$M?W3zAMC?;?w-Pni<P$c`S9$=qYq~S%AD-JKl?=-y%xLge-VE_e81lo2mgJ1a&USoj!rI;'
    '!*|DT4-eqw;rqR}ANCL5zY(vX;`<|D@k3yAfO~c%uuTc;@Zc1oy*oJB`w<>@Umd<ZJo}}6k-R=Udyi0GADxI@alCtScDVQ9?e2*<{%~@9bb0^?'
    '>;v5QhwopXKr;vL4&I;n&@8+Y2S34sIQ?<=?OSZ?BH8@_9XLT0#op2JFDHj@emoOD9=+W^fS0ch0Jq&&Zx6UF=-1xc-NScnvA_Fn_ss#-Is!1z'
    'JuJrvil2Wxz<1c<F8pur?C|J4_GItq{n-gTwxOdZXJy@=4^I!;V)x|m6lwDM<mesr5oroljtB}=e1E`Tklw-=7$}0zA5ITSh+_X>_bmWB#mdx6'
    'RT{<>25sf!V0ZuBfj=Igkoj7i?e52?Cp+RSdMG-(#J(pmK|4RE*=QgR$Kefd#uT`%a}wq0Xi8J>E)K<hJRDAm|K6p8Xhbtq{J%6#3UM0URI*?{'
    '>ce-IN#9R^A>*5*cd-IfuZXgXl^-q=A-)gAZ59pT^=(m1@}1SyFnfsa{4~2+4X^Xn^{usSe{FMnvkMje<LmDF%kH<^--+W8@cMo={6YNghH?Wq'
    'O|SiaI$lLl*}q%m?v_wqzW7e;0eUVVU*E;~6es}_X*vQSBkOE#ZhrS4UvF=%ZG8Kk_=$=D#i+<URBop?H-N%0>_>nPJpM+~d#k%4aq18mRWsji'
    'tZjV9l+DF|gyYF~Vm~ZGYyrr)I{9H9IZMUR1`U#oQuzHk1abOHJPQ-?b_$~Cm+5T?uYZ9e{~d?vuW=;a;QRlAiU|~g^3;#(j3A!HVe%`8lMhLJ'
    '2V)r*Gm=Di@m&f?UW2#?GLFa7B<}O5@1?$YTMQQC`h7gU5qZ{!_n=OqB3~WGH`6Q%!mNKA7a%U-!T)t~bFl&}HYjeP;J4d<gQsv*z;7m=&`S%D'
    'oY`F*-K!>KC%y;i_rtDH5F{V+eMo%Y&(b{aghc`PVIQ!EzY;^|Izta6L;K&aBEYJP5)}~6A*dQTJ37~44x)INh2!Wx%|3$E&&1{f=s94mdzxf1'
    'O|qOOSyoOmP-?efA)<!~=&86EfpX4LL>!bbP%t9QL9`X1ZIh@|OcN0CD96jDMFHAAhxR}ZegpH*?jOzAx9Q1l)l+DNyT_n3obDopC~1CFaHCO&'
    '-ykMp9OfT=afZ4nyi4OjE{^t2I@jp~(seQ+V*+Y2YM={Q2ab`$FA`wJ0h$cRt8D?p2O2FLwFPh%RGi{ZtQ==qbTb`=88-<GnF1juz&O6rn^`o9'
    'fZ|FCaz7?0D89+X-_sFTN8Bj)0~qz@Ct#J$&h{&D7iPfc1@<CBi;KU=;xU8DfIMSh2Gp-)SoFb$iuxZX#3(qpd$%wixunW<)DNd<!No9c$O;nL'
    'Z3x3yWYFEns&NBjn=z+_*!^Ngef#=r@h$=zXdsRhcULTj(283$>H6t(GK$a|98Z8YNx>FHM?8CW*2wApvuDf))5!p=xdm(_M8!g*a0W9$4rUDY'
    '#Uv!#0$2u`$6-KfLyI|BT43RTNrT1@_7g@Dtq-4(`+sxh8DO_71}KVwVo^9&U(jsI!RpN6MUuui&3@WFKm><Ti1RzAF@|X~nD%9_aDG-?%(EXk'
    '7dQ`;1u*f%7kfk<ZVREsD6T>>NzuqcxYPqEF=aVIGl}6Me<b{ch7e{jjMoh2Y&eQ!XJ4NI8u<(+c$CM~zt=d0!8nIMgNo^^iN!rZ+Zc#B;P*JK'
    'csOuGalC=T@DW<Q1(UJ!8$duoeI52c-el=C8RXyLSh8UcZ6{(JKM-ADl(yDBA}q!VW*>}2!Tkbb9{P`qhDjJ>>trA79f%y%5=`21>d`FY7ElC&'
    'kQ-N)K?rajaG_AKRSZK{L=%+4>uYOr5i`EVxp0w8pft?-x5tOar)?4?%rGJvpiQrH3YcVI|H7Ik?JGfnH0#_<W0<8Ei8Ul67+r}qOe#}@KSfh*'
    '1W0@k9k3vnqo`g}VX?6zcwj=B4OLYfHzf$vYKjKhrNsc4FE)JfUylRjOLI?g5{RS}#s(*b;^_3FoHi`~Gh!TMn2OtJLJvb2Rn!5qnjU;C0XH8-'
    'z|DCC%+hcma%ekmq}t{@s$u;O);GyD0aAg`1<EM0;<_0`0t!DQ)=4@BqiGZs^adyf6G7AP*Q04xBNH$m8L&F<of*qQX?fBF&mvr}p(Oz-rmoYZ'
    'nIZ9&3a+?Fa1w?$Nj#K>nJ=zzF6~|^eFje?v!DzMtbTtAJ)42p3P;nB1qv=9GnVPZ3zh)CNPZWuqNIO24#B|rT^ygjKY>4|r-$$04^3hC`}%Mn'
    '{-xH@Nd8@PIvqj(?#%y#Wl>~y&JN(+`j)@>E&N^cUu<mC-|p5r{M}ghyBqYpy}izrq@e0RPlQJ-84X$pV$)w+r=l;{8B`bK4}f^-uXXwTO9H`a'
    'mx%fQr73rmAECpQGXGtiDX|LrGsLdP|A^qvUTGPa9TxS@Gsbc$Anlaj#rfaw!&pA9@}EnuVM$RhihU(r^SgkRj(GP9Ap$~0BaFsn$?Ekey<Qzd'
    'FIS6j5a(I&k(vmG1^nJx-(Fi=^CuriEn#$|w61U{_Hg*l{{dLyx=u5^8qsl-E&KGm)I-0E7r+r;ws$Zg(8k&fa2EHYp!;G=b?|pJQBicjl#j^W'
    'B@!_3bFf%v@FIud&)d@89gWyR1XG2lat!s)uPUoChn8W6f=Kp%1vzwe={0_9`C@kvP6{-BsxlV|f`yL6*JwIMV8u&AA)OXEC_(WiF1{o4v%?%K'
    'U0*qQ4dY}=W?49yj7nqk5Hw~u8il+Rxk#?AuAu%cN`hn%g7E<+RwOzgD2ZTVG6Ul@>21JN5}lbUhE~%Z7UWU{TO3UgXgJzIpB)LIvm8d{b{dM~'
    '+3_J*C7^XO5_Dj@k;iNhP0-{{`Y~zlNjyQ!L4A;!as#6mj{F!(JancRSl4p=C$mZH!^$Z13)zAn4W@oSS<Pb{JI3eD@dx4Uec0a>U63KI3axYq'
    'ViV2T<JnmX>qc>Z8{tX-+mTqrOeP>7ey4x$Vlr<!ojN{)^d6Su;lKxKVNr5$+~1B?`_t}v*M!(X4f5L*EG|^vxnF=KfTMsozXsC4(&hL=Crx0m'
    'qvS4zr3991qoNoAzf%Vk-ne8zW<*PD#~95o*@wxrMDPPp@T${66A!(%FSj?>)~?tPywaMyLY+GW18o$C*CU)!NfMEjmdiyLsy#4k(`*K+?#d>}'
    'Rkh5+4rJ2=TLeoSb!D#mj|*DS!GikD+jr2#k5K|sG^_Y(Ca!X{Rg!)*iKBjWRa&*>GB8etU<N}0LA+BfR3uht^MWQtTZ?^TXbN0U;}O#xXrn-#'
    'RST;1eUd;|zW5n!upISmL9%5Ku}dGOkO*(krq<vBupfGBKIpu)tM94-#EM|DAqwyYDOxUvhainaIzq!lV;5M!fnbuZhW<z`q+$N=%VCy2C`Aoz'
    '35yJ+yf0EFg|yNGdkUw#ViX2ha@^MyMJr%CD15T?0oG6?zBanz%`2Q@BvoMXl*JQ}g$iVVH3O;%YzS=17%8C*FM#VN)1otgHR*jsn*^m2;YnZi'
    'Z;`+k!z>+(M%95F{3ELw@f!6E0zbpW6)gMW;Y`Hn1Q?>pjB-X_iPZZChDwYp@Sqaz*uRSlG+{d(&C#H1h=n~+^`(D9rxS0sbmWQ}Nn@i#y$DJT'
    '_9gM*QRCtrRlLK+d6f28&5%C=G;Ib$5a+l$m5bXBFwJ-ZYwYoLG#IeWOp_68?wR!0&;%GTpVdhYraka$6fmHm1Y~qn)0{Rpm1UUYYTsA$@GcrC'
    '@FaQiAW8ef8-G#>bevjg76TW7wvpDKHr3FlX3P&TTVZ~PLtGrlxd&y`#!tDs#<i6UK!r+vQnZKVY8(AE>}C3nYk;6B*bTwnErg}yo>C(O9onme'
    'GYn#O11ma$QIbYBT$vOS5I&zw*?C7^6X;2T!~>YuaeoT`?@TA`UZ8>8d}y=70%*{WavV<5B~0vS(7s5}=Q|47&sfm*BU&`*hy;lgj#-~niZF}O'
    '&vlh&{v^4%LTbQR;IbwlKPJ93#g@f3L=>GxV(P#u(DQb+NLOd+Gz&&qz}*9I7N3x0lIy$G=z+TW#pcyA?8*$;srdbIoLj;nzm`Bzf>akTwqIf&'
    'c`#tDwr!XMV6&i^1%l8w9Ir_f4RU>R5?GwP4C81_+;tl=L%#&JC)*PJep!bkDrzmcDvojO4OSZpcJf#2dip!E>q1!l0TA2_9U;$x%tI(_Y@)de'
    '{lQ*L9+-{VR5HDX8Fs~e1Uu#yyNcQx2*->iXFoCz;a89oERS!-$bmq^Yj)=1J}gY1n=G6W<=SXdLPxJ@mlqnrRX3Fx;EEllN^PwTGf6EsYoeoc'
    '?@2zDt^+xvv6Iom1SC4?ewe=G8b#tXTZNV#+My=c8ijR7zYlYq)cq)&CSX3Q2|~sA-;!hX`NCg@WRc|6yrHFC#iLQdK2g^70j{xvGz+lwBGwN6'
    'Fl+Oe1~^msr_}uNPoNtVk0XSfx{sv?a-1wum5?iIRYnR01~&p_X_s7;ju847l%r__K=;oQ^|6>y{oiEF0`5#HonTU(RNCxaIALecF(M6rrWt&T'
    'gzd$_$14`gtA@#Th0Y&Xp5@^%()Wj8Qca?Mxw~j6;L6-L;1;wSAYmS{VC8L`cB%hJR1LHhiUFZ~ErA^Kww1ZR)j^r5HtBdP<8bvz?818704Ci&'
    '(qNPj4;~P0X}1u|k^^4t(-@nx1~d>fu=Y!$TmmYC3RZvGJ!sm1wU%EdwWI)Cy|TS>=x}aJ=Rbx9q2ViiG`!It|0Cph7zSu-`YUqlB{!o;?t1RM'
    'J*-jOob*@9SA~sOPODN<LqCAySjDTp+{PJjdYBcn)yX97X9Nj7wN}@sIk{&+TcWkAAeyMS*A&znfD(Zr($}}LWbulya~1*w<i}mm^3+3geRa_F'
    '#Y-O*3ZNFRnl&zzQ%knd><bg<#g*nO@VIocY5q0Ye&u0PZd(~UM}>C#ACvTcgj+|X#3~QoAaYr=^d#2}?94~x&$W=eOHYRCwLmAIsIt-)XSIvk'
    '7WWO1``PrMEj+Za?MSfF9hVMqEsi^>mGz&abu5q@z%&M%rOo@EBc`aFbP@tJN*mM}I<7jj^^EJLTyD)<3`Fh;TEm`~zJ<zts)f&Jp48QWD%k?N'
    '`~3Le_1VHIlW179+NH>8OSE|ikB>XR+rHfWpj{p^lCI<~(UsZ&;f>q@ugJ0UpmkA#CNpGv+7M9|(w>YGRAx<l4R+K;@}ES-QS|9{y1ur)jkZ-A'
    '6-N;lqp0^u+ag~WpFu%T)o!M?)j20-<TJGvPGK#P^*&*Mj*QhI34t*>Dqsg+aHI|e?cgH@>&Unre580C6}(f{mjOK7(66ux0kg38X?JiJCVeLB'
    '`)HbhsR*mFG#fK|$GogUL{4XU5sf9HchJ`{&}HK#KTm)VdGFJ9cWZsU(}6#m8~kAgT;BVHB>jAme3n`jqs<y;Hb$R)!m1Z5j7LC$MKEIUrQ7B&'
    'V;K^)BhK-K5TE!Dev1eA{$i!UeX!E5ib}r+msbUi5|uxR3pq9(jPgnr=_oWW%*>`hN>Z0zyxiK{+SvSdOZ%ALhPY-Ts4x0a|K(uq+wO~vwXL;{'
    '7yavS*uU<=e}^x-YuoGLV9?#}Zd`9|cZXYp7t#8l8*P32?J!z{f?Ls=(P|Ey(ARYUx3P?kv}%kY*>qaZxQWm0MdQPELkG8un#YC<%^jKgF4y9L'
    'ac@<*N2UAi*30ef&8^LbQQ5o>qv6Zz?l2r|bieI~!xw|?we9}P^{tI>*EY8L+r$31n=d!|FT2BUH@DZi8!uk;w>MvQH`ceh(Tm}WWkv<0P=xbV'
    'b}#^2D0rJ@VSqM7@M9Df<8TsS%t&zB4PIl|LjX(G0Hmm6u=F0w(ZWBMG*NMm+Fo@I43!%|<sJe|QLy4bLZEAd*YV2VK8&-x*wJ=fs*Qo0sOSJ3'
    '+__#IA)=$%s=iwKq1W}-{k4mvR4U4^eUWGiBmkv?AJ$*shh#dQ%&6uzKB98LkKIjtI*xJs9KNpkYxs;NcmcXGWzd`c7QEA4{R9_FYH?0bbkpB_'
    'NpF7}jPYrk+Cq9Q<2oc7eziIUlk^%a!6><tJ7l1cR`9>aS=2%0HI?p7j*e+`Ym%i0MxJ!It10R;Fw}CS6oX(mMKdl4<R%Cxh9qSrL{^V_o818F'
    '6qm*?@>woH;5uy-U#t4K0wRCi!ST`Q;n~s2FTu&t(V6Jc7Y`bQ85aaCUq%UfEgy*jtaRQ57K?{yTY6UQ7RIcjE`nx#Y;cDeQ$6}%Fh*J7t+k!C'
    'TNk9)2houHm0nA9eh}}|B$8mLYZ;)Zy6Nv`HxvbPOkcb~#A66l=>@?c?L!Cjsy@ipAe42yixu6ahLP{e$=o~3rmzgVjYgB+#mZacZ{biKU?yZ>'
    '?SW+p<F%k6mdLe67StL<cQGkFur!9@l;UOj(?K}DIVp!w5~&rqCZ0qIODoBx_dxWZ>CkVBA7KRw?1+;mFWMA&MnR-Sh)dG8C_-4M1N-t}lD#ki'
    'bK~IXRGf%Di0LeB3ni5*@VB!G?aeU^F}Y2_PVSIN#@_#N_z;h$V_BpvHr79)aS6gwyvb-8+ZKR$7K$JDT@8SMMR|v-K(wjTX)Zxg&HN*9<|L&p'
    '_3|a-gyDY^Sgz(Ss8C>tM!h<#GKfkGD{pfoS5>7Y>Adv#B+(5;EX^Az&|{-%mJ!0ha|NA&$#eJvmMpBcq<Fz)@O||_vB}V*`w&+OVfL|2AygDX'
    '*%rS}A{aR^n?Vla3YvR`LkmDMa*%@M*2%|d3X)>Q>=yJThj}P_M}M$?xkpnNORz0ZdfG3&0ZRCk8>T?KgM>`zV_8PkkN7n-GN-VIVIk-g>S5!4'
    'Wm8~xaOUuP{@pSF`%22w<^Ty~j|%fJw5f}}8l1p*%X3*uE0BZ%JHrCe>%ou)0Sd1m0G-b*vMggABi$K6089EJ2&6_~Y413)Q60u`pV+YVgs=Og'
    'B(2LgClqSdR?aHjJc0T_07@MD05<uGT?r)+n11z&E0<VPn)p=vVg)odsMHms!~NT^xEYOua*JN}Tq21phf-&ydHFMoWXFmb?EuNwZ)226D&C}x'
    '#}{O{Xd%lRmVl~uDqkMsOSv{mGu47-C6r1si_X~Td8Y7ol=fjLMrEB*O4dyq^eCzL(*o_Go0+Skqy~wSfS}M0xB|u8KA=6cws?hx8oiP`1Q!YK'
    'k(?fv*h!8<4XRko%cIe-&Fffl*6yJ0ZS7Q2=4*6{(3W}CN%akv>NF%$f9Gfxcondg4!Np^ZTk6yCnGS0oAMG4g6wG^l?Y(mUV&0sCDlUciNpQe'
    'Y9|Rm3J&re4CscojJEwdP~O)8I+nbaRqr~@gLvSs#@}jHqB&I}ca=#vXsj|#K3Y{Iy_@@hGRFK0rg%VXUiPvAm!)9D;iQdWDZ##Vp5WjAo|8l1'
    '_aAs_fB!og>D3e95!0q!^(wu_SXts#e0iz$)6u~ByWP{jUvk~^8~q{I9`ba;KO+2bDW1D9uCbNX+NaM}2?}ls=9Z&<MdBy4R*jN7rv7)dc}{;W'
    ')o@Dwb5wI7a<11w81&jvSr!q`d!lP~uuh~Pdca_2QWYqJf4^Aq(O>1AXORz9aX5)QD%di+?jZMN$w4SaluVFfBq!<7mRi)R*?S$1@<<mgpd6hg'
    'EgFLuD*_@GAirjQ!&!5FX}-fQ(7Khn5cxygRs|!+TTC73Ld$AhBB~+rJwx|th3mP>6J|ax=#znG&PSHoyE)0a+{@1^z&i%m%T}unxbVZtgn*u#'
    'IPepGX<a(%86?!onkDUSj-teK(&Gtzt92)fz|u@aQN4|TJ$9Uh<~L(-Br}x7k07si;GYzg4RIl`sQrqTlW01egDk$>sR>=MF#v1}cL@My_1Pin'
    'B?>Kr(-8*;?`paR1zcgy4riSL%AIQpM`Qnrs1-%n3oR*9>{NZE9z$QWCdf4|uBXEx8P{k%wvG8^h=*dQ#y^$RY?5&9>BUVpO(hJ5cy5zf_KRz_'
    '+%?Od*fOF9)+vlxlAARhTw1lHrY8$ibNpRBd|c1e=td8Y0hTzcab{ZcV8a+U4R(!r`QWykS`1||2K_X_)0<M97Acx78Xmd`;Ux+B4zLDB&oaw+'
    'Xm(QW0$`tkbRD3JjLaKfsHIFJug&E$Q;y6XW2eX=dYq+))yUR69}lqS^j&tIc&gs{s_zLG5gr~kyon?M42sx<aL|EaJYvr2U3Bt2y}WccqD!8('
    'ZYsBRi}n%&sFWUAbGL7H#7Xx<SXvS7J*K6;u#s`y<?6<Btj84b*+!-WCA1nRgYROcdM$Sj%Jrvo`h8Xd%?l6;_bUOM<C+&=#Dkpe)bfZd#S`*7'
    '+Fr(2RO(Xa9BUC)VS|S*X?YmnIgZ}N$|xGrD!V_5LRbq<0%L8MPePcjJ*C;2jjy&tN<Mq`@gDsaI-jzI3Ed<epc9$B2}ZdJZ78i#)GSp+-}m$-'
    'saI8KEwtoq*W35Nct7CCnJp-OtQC*k#iNGeFb1yT%-~gNdwt=gKcMe?uhm88jo0!^rZ8AQe^|f<^FFrSYEo3LFPm?iZ|8w2`AO$=T!qRg+$P2F'
    'rwMdrzZt-*;k0@th8HWJ==Wzd^>D``R7*e!18tyB<rMkcB*dDUC=ujp|2fH082MsWQ41ITl^$_u?oxn$4@G~d=A;7_a_bj|rSyAi$K7>(xXX#u'
    'ngbc9;F4Ib=JSy(K|K;l77p`})Eq7I;8x>aF8$4kKyYdxq+xVIkaBJec$cz@^oyGP!{2+75d)VI61R3lOxzmNN;phz7<0{K)cS0J^f?;@mjj^_'
    '4OG;nbdA@oL8@oyHOno)s|uH*anr4=&K2E)sae(<k|+2duua{)av@T;CWHT3b8<fF26vSPnR69@lQ6(@M#;@1mL%@x%>|BQkTFP@rxk|E&D822'
    '-<F%O?J|@wm}Jo~e(2%eah;si^J#b&Riq-w6mex}gPhL+<FONc!&ftsc7f+@uc3jN@UDOgS1w1(8Kq=V<zO`LZ(};GNgyobe9H}Hc2~9n^Dvb='
    'BhOqd?j}Ng7qTMuZ7@(#iwJU{vd2+$HYffw`YffG1^?;{)6@_nUhPmML0)FXqr*D{&Tk$Ouo0@g2B3>)64LIEVVSCm<oB%i+h!Kaw3W+-b92pL'
    'xdvKueOU%JW^+2}D3}mbPvTi9w8b~s^*1fl3Bx$CfMb>@nr{FbzWP}d4oqi%1y=`ov2uS6S`UHJ=_uPl!7qLK<CTq=y9hjIb+9$tk2y(qQRri3'
    'd@Y?~rhYDW30iHJ`lhXdR(%<7fY!017}y}E(Cz@Y%flO`uFykFvEm^{kn?6bd~MZzf>@H)#DD;8nDCMhH%sJ#P;Hd(qqa!`%)>2N+c9&9FvNY9'
    'Pgv?RZn3n`UuZ97*!74utiu2E1qL5!tSP%wwd97Odae13R0SQ;=N7!Wuc(3axaZ3YYDxCrrV&oO+ZeML%t$&jg?fsj)^b6XL^E9r15*D{Urpa%'
    'xnQ^6la&Mmh@&ZMTJZ3kMnt=GSneD{O6f&S3ZS!uOVIGHT=K@hVh~ax=xRtY#ueMc(hf;nHK=|)M}df=3BPbp+#gKGliZW7Nr;zhC?rL16HTX_'
    'vg+miIF|b_7P5*ShDYByIm}MA32o$0n0<=Zn9FI7N}P4C|0CMsDah_;;)}U(T=K%&q%?L#bSvW;H$$Xt8%##i+&eGf;|!&ufea-CPNWzdU^%ck'
    'M=>%mNJO&oRlt_b=L9VeQ2bhUs#o9ElU5Qo9;|7udi{NOr;KelFALySAx;7rhtsr|VJp;El&C-+Hzq_mGzXTv5zrlvxKPpw%mIcvzkEBS%(?P`'
    '{BhOMkAhN{UDMGfJk6Mdg$#Dr1SkhMnx>R5cEziMld}pkr?}IzBXMxmj;??@g1M)xwj-|2_uigfivJeprw7NES9)2yD|uOZmrILRWv#hJoHkNm'
    'Tb0yy+Hv?58$H~Y$%Dy?Lj%gRCGTTS=1U7Nzp1^h(cU79PB9Dg7}K+ek%o&gj7XWK%($xUow8><YY^zA=liQSc;pIf(y!@t2e59j1wscn@KpOG'
    '!yM8!2Q;&)Z6uU)-n1=jeI*B|XVrhMf2}*AB$=fH$e~?K*!t6R1HyBsF{7Lfmu_vTu+mnA6ENo-;=<aR;)ozAa&|cI=^W#p&S@J|j~KeNLWV9a'
    'SmW>^fc3)-Mg*1O2FF|q8moBepOo=TJ9dB*g$^5U{QaZg{n1%)yt`iqhO5W%bX3H!U<lITQaDIiZ93t#nz-9*RV6`*{*^g063|wZRWj1Ugo4qq'
    'Smi9;sL`hr$`hVNFkE!(nu#YU7U4lX$qWsobXp7Ki#3|{iM`JW({WjvR-qUblDbSnJgC)qMJsJ`)$*I!LB-*!9uDOpKcs#n<qMKTNhM5TE9Jec'
    '0-7I)5P?Xy+rj_*j6$%;rldURjAQtb!uv4cJ0AX;CWS+I`j6#EP9M)n0Qi$3{3?}UGw$1?_iqkP&xkYBSw76^nirKY)kk_6nD2J~J9vBW{>|Bs'
    'J33sAcRU&I+gb)(HMe%eeo9gd`sj8rR9iD@m~fYYk25N6R(^ARA7aNvEk@@XoE_inAV4~2NF;(+rQbj)-$np5vn&Wa+vbCC$qr9(VoMydXAA{B'
    'r#b~zn}h=cB0F;w1l1m>tUJss%DEbR7b)?iWiBGgmcaJFUc)K!CroBa%!GiD=K$sznUrD=BhI0J?|(YnKima_CmWPEBM5a=q#sUXR^WQGS3mT`'
    '3xDfMct541BD5!zTPW7SXf!&F=!hyU7pi<J42TT5G7&K4&{cf7LllMbY)%ptz$Vng2W-*$wRnq>rDSq)EeR`k$2{uX$Ezyd7&YAye1L-GX1VXC'
    '<S47-+IH^w+NB<{82u)t9n{AtC=;k5x#M9?)F5gNGGnJntS;4%R$MbpINx!N?`uUot@__w6~5|;wbncgJsrgAssly~ES_#lJac@~`$DN-@cVUy'
    'rVUQ3UY(3z&9fP(OAeA@tqMysTKEvto#L7Q$Yx?7F3Ctdx}6lhLG}8+zB<S>S4CTVSBHYO$a(h?h*j~MHQ;vmz|$JGLJ*MEKu3FOLrAsVve~UZ'
    'N>)pFd{1UV(1{N6djT&uQ+843mZ$6*QdYz8`4aUU7+M4HB>d_Funcy!$dNg1G)uM8m8kFD)*3g*%HiHM+p|t@XelPD11raA#@Ps$&V)kQ93FkR'
    'kxHTWiQEv?F`_U>!@ga8ns?OEag1a)nYA$|%%nU}@t-9q56b@|N|et>1tgirVkBp9e|BM#XC!-wCixWBf^4{l>UgI{$BG7Q)5vofGMI@EPit@J'
    '95Wpv;Ac$yK7kj5a_NUwy{bLJ0|&Sml1H8}?<y27aC3%B`@Vl^hIt1Fs@j~d0H>7F;<CPn$K>ZKOupdc;J-c`o*e84Z%&Rr9G_MpJ^J~4FxM_t'
    'a!gV}Z3fzr7<nq7`gRva{O;E>Hw`);6;O4s+y*&KqQVUe>wd1znFe_9#bfd#pHSZ=OK|srNmSnp!(}X)h|WYyIzRNINg)pD!vQ6`4RfZxPQ5S&'
    'tLuc%SW~ImIZSmnoAQ;&yoX?R$+=bMCQ2p3$GWSul(4*#Hv+`DN|5IpM-|yDBm5LEd@ZhIf~ElT&s?Ftyu(w!uvWN^VWcppP2B&;cTs?i#vmS|'
    '{*;rZ9a6Txd^+K@<_c=enKXcFWZyU~;Sf;9dkn<KD4OJ!wBY@$5$9Z|Oe|^zXbuDoG5Kilbb2|UZM^j%z{BZu9vC=qGDV)y`=_75m7{kyoA9VS'
    'wFL=I+8c3B3D-addjzEvIIS8>{DjL6W{Ba%%E1GUH71|IR!D4B1Ky^Lv@veYe&VoogpHwPd=l|<bs$-}!VAher}!STZqk$>LBrK|>Wpt~;D3x}'
    '?3j}^s;=TId-k8}lm4>h$2n#Kgn|gB<qG|&iZ24JCjL@jRbhi57c0{Qvj-(Cg_$h%6OLXfngyo~Ya7y;!Y9LNk>VCB&}y!Rpruf#YfHS);rIh5'
    'tCIPgcqz)MyyQgBtAtJ}mzWXOQ32#^Z1N009j5c6hY5hk?KMw3%y00hRUaryo4j6~MPI|~_i6DOFOMiWwdylL8{)?u@yUYvSqr^V^NFReRj{SB'
    'Qo;P+b26(e1<J=Pg`SL|8X{gn^6=1xz51ZBm#i<%ZkyE>OwtcYDd{S<_@sb-_JJQ#OfZKiw9EXwlzWny($}b>)bvP&&uBZCTA#ks`h<*A_8UGZ'
    '9JDY60;*0e0Lmxm1EGA>pJeIJcpgOO0=)hw<+QDz6!gzJrf%y)J7R|(H<*qtik>U#=oSTeTu`#$(9)J8)HY97Si+QQIEAUR*Y9FZgG4dwG}ZK3'
    'Mg4(+gGM#Uss6*gJ%^8%_2X$O(SG&TERM@0!nNKTdc6qoQwvM-KaD5F%s{u!&&2tf&82OcROXgJ@p(4cr_Uz(+CXR}tKlHVq{!N227rOnLta@?'
    '$T@Nc{fLcAsOAqMhQZ0ihI$W}C}n2WB;^~gq>?uWQ{^o*d#ziRb9*XFHS`iY#E12AG+aDN!W3MCb?^~&gWhKeXi<Ng@6Hl~G-B6}?6|;d$YieC'
    '+bA4V{f<U8t|*P>s(G}Ro0&mb({pHpMV>PJSZ@&>WGcK5o%d1kl<^Kw67{S#1TX!~H_}RCO>wa@j;oLJnMiABuF$G;5RxJ-6wu`ep|7@cF_w}I'
    'IiNX1U87XfP`9L;Bf!mF2SuD>X(fuJk!71h+DZpieGjt$+3LTos|8X}J;lG;Ct48qY|90Q1k9YgpwQ>GEtGrw$7lgmK@W$3iQTp}XO9-m9zSTw'
    'U6jShiT25m35fiPXC7%dYCF(n&@X52G}GPl5SJNPDfJ|+5HUD7qI|8IV&j9rYEWqZf_*T&OV4R7JIax7CUh;+-N+jiIMdJeD@tLhotfe_Un2c%'
    'v?~Hj&s?`{$#kMOcp3G$(#0_S<ua#Cmji;KpnQ+^cEXdb+<zZyQ}Auor8%b>Szv|k)hxHtZ2KzvTqz4aGM|+Opu)Q}SfR9TPb`w3O~wXy-pF(G'
    'hedD8L`jY5JQ>_5e}l)Z#Fg!6PCdqeA`6h*W#dn<$mZrsA}c_@d0D@BxvrENi+iO(V2E|SZ8Hb7Lf~DPe>|+N%9QX}7j18@y>sKSsFr6AC^=Pv'
    '%I2_#ip8vQ*J!S3JqfnvnWpu#$^zBlJEf)>IcBTe&X@#SUAtErm*q{0@{Wry<f~rFM_uMUqD+YBJWM9C{PE#iRhCjg%}e>@PVltYyDdquN7Rx$'
    'Qsvazkw}`dSbtkfn5ci`p{R5pr?IOPDKUVW9v+X_j)haJK||L3C-!KnS*3iV<MfVgCUmRfwH3Oe;VT(247lzMV!V9a6Q_(6eR7);hynWSm!Izl'
    'UOAMjQ=P&T4t%llolr?#BtU7Dv>uUh%}%pc5~f!xqRH<v%^KNjOoECrbc;>0KxN!<;TmVOj4XGQs!bN<Co7AdM2V#YHR@^5D_XVe3VFnL8F;2Q'
    'bfS?#W54X?x@NX1o-^J3>K!I!>+y{A>F5QUBl4gWjOWRlS0y`ySr(!uO(uSr(}PD4Ku)0(w+HX>ij!hvy(Lz~`qq}eW?;?kXB_8%1cXO0TK*);'
    '#?vD5*4tvOExIiq%rDUK(LW+@O*d8XL<)9MHd^bcWrr$s`L3G^qL1kl`?p|d5w`HRKbhjcqi{U&7z2e;w=EE>md^-FLNL9sBTNPEh<L_*1_KUV'
    'hyXFn$+-=5nZn?{Y=LDDeelXGVOqa%d)LE%0=n@zU)(*43g`)~jxX<Z!4-t;?NXhc-#2P!Vb?g18fF^>b}_oCDZAPw4VA`YX92aPl(5PzYpfsS'
    '8q&PrO<sz}(M^TC0yFGu>kNHiKya1br0WlA3sV5d8$sr}0FN#3#e8Lu7u};N>zBePPlc|#tW9KC=v=G22;_}X_By37uC()2a^6Sh)0MNGMJsQ~'
    'Nd-;ZSM5Ww+E=-beVO1Ny&J*$fW1JaEdDT&*3c__y}o~u)He<nf?XbXl~3a>-s=87&Bm{%30Y+xOB7tP++^XvSMQa~`%kpK;{KQ}sINAQVeIe2'
    '43<*GLt8wk9ksT&jzgGt(Ztw=fkHm+?e*|O^*U(dUCT)V{b}O^rjC5@?L2+sC`{-B{y<@Z`-Fpdtnb@Zjsgl&@g78n>VXhS@4fZbqPhv5HyBZ{'
    'tX>-=uvdb8PQ3!DfT_65zLE3+iUhciNo1)2woAkmk?Lnu_`Q#Y10q=wrQ7B>D-FNrMB=8h<-vCMsFnE9u;F+IGLh+yix_p>@tl(E@IalH(e_C3'
    '+N(i;aG{_E9=c%lvC}OvuKY7$70>7CusS*co|)ft;w&6zI{;M!<gSV=z!QYVU1J8x(>DGknLM-mI+FQx>|qP2IzTgSv}_<a!X^Rd){$IP(XXxJ'
    '(ym~(<>?`ZI9LasP5gK1U^*(#KsvJ5D0ZVT6q@Hdf|Mel+l@NgMt&JP*NL6`rKA<Ke5|}aQXT=Gxu;rd)zWq{fgJVz5oPI|ikcLv%QEib!;Tzr'
    'j@>1bL4Q%-U9QPgpRQZ>9@=_8+ZF2V5*ahs(Jl$>#}CA#sNN5ma;FpsjggYrAYDBj-$lE3H^*r@0(R(vJpvyv@xc1PtK^I(aaye}NyZ0?kt7T~'
    'GYQoGG#s54Srm@nVAcvJede~_7Bd$*6bogR)1UO+$p?@O59_3d`nS{MBO2N`A9VpXgEBKLGixrR`r3ug&fxnDe>|+adkaJ486)-d4ub(qA;Nqx'
    '=F#29HNXR|Wy^!O?53NUhik_)Eb1JAwQ~T|0l{z?oE$R;UD{&SM2pGtv*Q^z0<)u5dZ{@=;j*=9qM*7;XY)$`)40?~n1p?RiPYAR^k&AqdN+M@'
    'NJKr#;y+pHkcHX>BAMK6a)@KJ9j7ccl~u}lN3N=QvJ2N@D8H7f#)ZwNjWT&#uiI8TOL`F-wEb*}wC2wxC6sokblc)O3+1;4!XB|u%>cKGbr_$m'
    'I*9C#f*`(Ge>~r~xnwD(_~S$MN63Et$H{*EB(gUNDLi~*%PBge^#^4Y!VNeNl>{iC)UjU{86K9C1zJ|^{9`<!&F3MeP0#?sP-?r_3|toG<7Lmy'
    '7N3>;`dIwdy%LkQ)=*uuRS4HG)hjk&b{%Af;9j(|QWNIv{NT$x=>pHVs$RNyP=#Mz*;h+lU0IRpHN~*3CMahbERVLjFxjeq*0;7>wG{3phRVUw'
    'tMoY@W3|M<xhW$I?IxRUaTJbxLPJI2blE&5r;{byuWJ+A72_q-aWF#Jl=s##WG+zc_lOJ_FJ!*KYh%=f>3k>AoV1J;&*kVDf(1qoS7mMW0;m}b'
    'F-kd)M^nCth~7&Dwz2*Q;J9NE&;mNhZ{s0ueh`={Hvca|d}%_!Ecw4k2<r9%LNunn{nw_%9I;cz4%U_yCJ6ZU27#lsLqs&Ggm0=o2-R;J5o{rT'
    '`@wU(m$nQ+>zcN&Wyxs!6#Dhg+ATk++WhXPqY)nxEB)3-8E`{1(5~<?&uY~!`{6r)DBm34{6i-eG~fO4;am59y6M~3pZp}DLghZb^1PQP#->Fo'
    'vt<7LEuiu~D%SUbL<2g;_l~cg+TTBZciO*=P*Z>n^m>%uzoSGP_`DzIeXt55tpML~E~<;Js_@qJeS&GD<I|QLUP{Fm=_vGm3M1NnvhOyk?I56L'
    'y$g*2xZ!CaU#lnuw64^pOKBGJHvF6$laN4V>gTq2GaBn1kh{~h%b=>es?3DnUw)6}e|}%(Pv2R2-xhmq@nc*3?A}tDF1evn?uJa4-AS2gRpj=`'
    '<91GFwCUj3D?!NCCc<8|Lo%~BM)qp^3(^pueTh91rn#iP9{prc-f-YsV_^1k{=Ul1qTQ5wXK|PUs>dTne#qLW+2v{UZjhX<dDTk1@S>TnU)v**'
    'BaT}&H^2(GtFPd9<S)ne1yI}CG?`U<B^%Yx_3GzZtL@w+sh75ov~%61w?h_s#5BcPTMTJX*%>#-PnJbDpwyx)D8pdAZ+Ni3X)m!)vJ5z!CT?K%'
    '=E+lbPg?(4`y=(ykWdzmgkztiW*IqmO`7Z{w=c-BpB_WirM64Jpj%9>I*kkn3n?W-W+b^({LGx*pzHq&?yb0YS9H|8*?LRvwHU7-)?0sk=LG=Q'
    'vP`2Oot66~Wr7ykGkN%@?U_7c!=!dOh{XrX%^CN;$#VN8c3u)ZQbJf^)O45PJGC%w{Bpi_+13{g_8}s90<o=;SXT2ft!eJO)vf9W-PkKBuYqaJ'
    'sgo_P<2X`7X=w)!I-U`%R_GFS&mNfhcw$5(fNDJ&Sx3cXr|%;-ola<pTrT?OGUvi%h?<vxLlVr-y3b&#?1C7y2}^dc_WRSqx=qe9A+mImVi^_w'
    'u;#C|T}P$VwVyQy0)Gw&$P%d4Vl8Y%lO_Q!xWg>Qlc6fcI=_$eTjNG)BIU<;!d{B7Kfj-_I)c&AXfiL;qLYjidQ!IGa9C04bu!|X8?8nDX8Zcp'
    'EDA9GMz&saGY6VqT6?O>>4=$SC|SM!)z}@^zc`#b8pt^%aR3Z@a<1s=*_N*Q(BfW&{%pzEiUk|;Yik91@}WjvByG3Mj8XGOnaA#g;h@qe$p&<U'
    'MZzJbBdo7c*xG@vcUgHzyUGBf6B5@g8T_DwfV~YqK{AMNK~p)fY7G#nqVYAlR(>rLE<5_XzGR<~dRD(pESKxJ6gYaL`~1q2pY*l0V@fH$v0sDs'
    'nZtfqW!<2+iM$Ks(+Cd&tdw8KZH6<-GsIiHvL&WOuRgiIYUn@A1G}~{i>;4Nj?^2jGIUqRziJ4&ZWy)!RU~)Zrl~Q#Gpl@kQ0m3>QR+Irm%~>7'
    'q{ea0@eSpDudV{xwR<$0EDTO=#%>U6msz9RZ9Eu637-yQv0T;C2gxp12`$g~)LK}j;?G*m1$9YkJ+=}Fna5XpI+jw7JdP|RAm+5BCV<J^%}KZQ'
    'Vbt3w>w0<#qbc?RRdo$e&rS4e_N!6Za69ZDKEm(tXGTReg+!Sqbsw;3o91A-tF<8F32Le+n5Dy)9D7DK#%1VIYf;H2O^Q2Lh6Y^<6mMuPQBdo@'
    'BR=N|fl(^1rmc?V=M-daJ5ZO7l3D<7jvx(2{KpWT(ygF}*%(Dl(|5#Xm7Gz~`{$MODJ(0&+nF<vrkY<Rr0U=ne9YqLE(%6?heV-YmmxRVq;J;_'
    'LAr?&Tp*)@C|3$x;*I)Jt{MdI5!0zbZDm9+F|4IL-#5g&D8d5P!e--`MUW8;260}{T}pUISOr#I{~|%|G-TFP@O%xJ&y~EgLweShKZZ`2-HgMB'
    'N1}t(y~p5?n|JoiYaw@@Kj%ZRyU<2_JuS*(uzTrrBvZ)raVt|u7<g=~%+qZ)BahuPYmLK?QP9ncXcFYO(^|TFk`&tGcewk?Wl#v~o3?z$a)SIx'
    'w+(YT96o;*#;f_D1Kqq%F;F0%UJofpA&Oa5zdZ5jY>f`3;z|P_a-8Dw1gts~J-2c=n!un0{plF!Kz`RHiunlGIs@yhr%`Z*2{YB5kiqbNP+q!+'
    'oTli>Plm3J6Hk`mBBLsTN~JUVnojFgxx(;C#yhAy^wldp0k49klHwQ*8TeH7%|c&`F_;4CRxT5L_~NSK=p&kE$af11hW*85>Tu}OG|OmWR)oNK'
    'NBPrC?#QA?vs;?eKs;qjz-`5RE%9Wv9H=an`r2h{YfZ-Zb)OL7citRq`<qu<X5F>*&AIW=|Aey8MSa8Ua``AitB<w7D2K_Whf2*|9QLpajv_`h'
    'RYRF&0?q7bs>Tv_cvWM;dGS^Y!>i_7t$$JWe-C9*qWrHU>=&k`n@%Q^uD@JF%BJYbe{EWt()f#qS0dfMTzDlxv%@Q`b?8FXjxq0lsrSFs`(H}v'
    'FUtO<-v3hXf2sF>NA=E`naU(=e_Wr})3}ib)L)f3lUM(4T#vxf-{_z)4m;FbVt50&0{D5A?^ZD&$ZN(kU%I6+l&$7d0!(qQH}JFyZKHM&5M6?x'
    'W#04!bmTkz@Ze-Wg<<3KjObCxO+-|_EQ*F8kBbqqpp2RFJfgle>XGlg+{IRpw9U!aZ5zI|<|fx2P132FKd;FRN0Y>ge_oTDO-)i)zhIN`O_FAj'
    'I@n7Ubp9xDz<GdPHo7OA+04NXWf(Hd@2%ET!K{B7FdJV6%w{8)FvB?~Q&a>z+4;KryDs0FMEkncPmm_sLm3QFzS2IX^nTH4InWgq@#RHL6&dXp'
    '8}^IMrJ5+x;3L<BU%!>V?#f?)tG~7cs7r$d159ThaOz1QTy-qU%lPbKTyPO7Ur~jH(>#yEWQn$%W7Mk;nKdm?qswG|o`qvuW%;AKfdCu)j<hrc'
    'Ht9`f1(fBr_kNLj+Wv&S0OI=tT<75yC=R|`z(&OCuLOT?-;=fU2-o<ufzRu#?PTbS6^sunX9Yc^s@8Dm1G@gY@wA31P=D2R?4^18!aT0pT*y^6'
    'zcdlC<rp`EcWhJ~93suLA((v>Xt6GyJrf>AY?r*EQpBPOX2c=2M91MOunsf!>gO0eyWO4>8ym8`Yc`OZqo{Y(#tZda;;DVsIa|wROBT~PFmXp-'
    '*gKG4<;~QW`pEJ@E?K+0&0fDdtO8uWLQLNQ)S_6c{J9i-Q*Yv&_^>K<v2t0ZEVUFV@6+RTpE%5e0MESL<kf9`+6;>xDu6o;?baYc+viqY8=2aS'
    'BFjI1N;74}<_>>$MtaizYdJI={NtUkui?=GdrQ{G`@id_8Px76C6cwbob4vd<F38hN!JBZvaDKXjVh1+qU`VXfiAwZFHHjOYe@)x^qP1PuZrXn'
    'E&B#ZeqCFXl=BUAh_mUITi+~Iw(9@9Dixk(M(AcV^@P#p7b5^C@YqguAn$~LAC>{2^+>@n(I*~p<NuV#og+m}w9L~=Nm~Sya)X}Ap-C~Q4j9&t'
    ')zDoJ+66$4(=5Tn#LOM4uLL$4qjpc+LDR%Vi~E7a{?u+b&~hz7+<jW2>iW~GZalr}=5m9ePfs-#8aGqwZqDaU!6jfUm`pv;Zk6K~=jQbwmrA2+'
    'Eot{iw5+O?fn%CS8X$+=WC6*-<fFV53u|p!pKDiV(bae#cf=hZOKZaqV;1T7s^e)k%YDRHeQh~L?>;xTOdgpz4dbXjqFr7v*p14g-<YjDAIQ<u'
    '$G6JYU+Oj$MO0xr(X#B9_w~>x(`>S<7-46IU1wh=QvGy}PI<kEEsiG{-ub{xLwylQmmN+JQ#ADS->HKdT;zvk!7^_(LVe=BYfO>3m#=B~YSVoV'
    '+WJ*x_AN5@C|3~i%8mLElcy3FOUmMO9Z`hY65qs0$X>3t=x6C92Z@<xgB(m*u=OdWos=<LhHig=cVsV1-zY8@*`pcYQRnJhiOxtmG8h|%JB~*C'
    'n5)`_0KC^QFr@YbudNwSKgo8-<}ouw!(mc=9fhuK!dM-nlC_`ld0q|K79!BxU1we-byHXkHFG6f-ZA2H1UzWgI0=SPi243=>u^E?%&&S*GFVE#'
    'c2`uLM0k4?$^_X^j+eC2GDaXJM2V}oi)b~S($VJ|Sa5!FqbBca4YdlNT&R8)_hAlW58aLizBs#;S0P|PFCq~G-Lov>8-;P~#4%kE!S^nRD}7k2'
    '(L_`OWY#N+IleN6aNVyid(p=7%6IJDG_tU{6w^Oms#{X%-}*^-y{tZKk;!kS@c>VbOJ_mZ7v}`Nx1-Igt|Je~6TRd@Csb5%dMLXfFLIG{c1{9p'
    '9K5Wc8FKIEw;}Y64!u_;?lJidOrkF4WLz^2@`R(IoCF4vVs%xlYcpqjttpxt&mb~F1+bLu-YWr`ZW<i`@TE#`1ZL1pRMs)a2k`AcwHlCnT%~c5'
    '^BqpYatUSO<2i@x?_9du?Zd)e2Knb;MIdaIU$md*^n6)Q9f5P)pv>RII+1q=_mJ?A1@6dMhpn|%lf)m&#3kX#hoaa}Tkyp%t=cHUM;=3LLyae;'
    'Dlo99b1!_>nHG#CSd6$1MpS$%Z}GXt)M8SR;D#l=&VvaH)$yb#^^3e&sX9H;pjGPk=E<f!b6y-DzV-RGn?wGN`lr=;l9arnCay?46)E2Dg7?GV'
    'gu2ojy=sdqtJy2GPqAeZjCf@=U93a!TA%029!cFm-ZG*d^J3sS$*GP)SYA%;ECc~64`f)*P@}Qt6L1zz##EgLvQ_4T+N|Y+S`K;W_9EPoGK*1X'
    '`YnYy6m=o4@ADoFhh%x9u2CmGut!%>g4g-U)kS|Y4N@+7MXHa#4jPF-zIn2fWVPTnO+Ok0sDGZ*d0si1`jEFwCezy8yHadT2cb`Qc?bOv^sbc!'
    'McysOw{M})WRIkp#ilaUF2*h80aVr=p1GxiWe~}`zDR6!FHwqZ){bc6PJ}#-Bhk}=IC7kPO}<f)<K<DfW&hLe0pATJq?RPxq>ZT;<3gl&Q3mht'
    's{-Banlr{`C|KTtdY?}EVO}_VGbS;mm$<>AmCi?|uc6EhIxB)zQnHk*QW6p;p<b^}ccEojjfn%%C8VX4n7^b`VCcI@TB=|Yk`PRXm3+^2d(uQi'
    '+L6^=uy}j$aENBxUBoEbqEl;I5J#cyS#{>KFWwoWOrN+UT|67{)nXDKdHq*~tM6*BMsYfnHizb}u{XJT?NWTN&kGzR?eTARVVDAIJ8nuSuF?z?'
    'dW8m!r@;hk%mmtM%Pe$txZJ@%fbUn$_%6v@a*+4vN;EN;PDXJb(~&D3oeuS}{baWP8P&JNNsOt^E24?HY6I>NIp+<%vrm&&4snvmY!aNuV8O%c'
    'vx|dJSwiKn^Z2si&M3;%gffZE-l{i_@|O!cQSpra!Oa&Ak?MfBT>ss@c1gBJirAX$ti!Z4hxVvfve0O<n^?LoWthyd_3&|_a&3aehK~01Xh;@G'
    '3|@dWm|@_l^VQmr;Bpq16ml<qRmbHiEVKarGo!p^IcPKe2Xkyo(Nq^*&SDOU`?cT;-c5vYMpp358p;<17MzSPC`}&IE<)k#LSruP>Jnf3f`Wfe'
    '!}Zxs(*Da)&LR7kTYfDq>s`%~$Ty@r+3MWq+@BM=>)^^I30;x0ww$v>iz7v(^~I4o0XyGw<Tj(UScWsHO3Sw8yYk^UjqZAaGgq3OqF+lbsVojx'
    '+zh5VlMhnV*EZuUM;InrGG3S~uXSJSN0^jKz_*d8c1If)8(sTKXN9P}AFnsE<%?r>cF1AjYbUQqdX^STLe2M%u(QH^r#VTi7;UG=#Idw9Vr$sf'
    'XD0>JFRrfHGPMYvI?z&iSS#R{I1YGPA05S`G@aOlk!xUeeWsCap|O-1zv8oJR4d}uYqhP=84FJs7TUo<yPlE@U{HTNn1&<09H$+?vbyWubGZ_X'
    'wq46@g@~OkQLey^oIgE-*mf4eEY0&EEaYXyGQP+q0ct@=J$EpbRgEPN?Mc-RT(n5rG;+H(M$lKUqMtn+ZBbg0wdPueM!Su|0uS_|fbIJ-q%F7^'
    'rPrXf)JN}P<r)Bs-(UdaV%ENmZ*E~g+aTMsXdI1Z0<Xr7M}P>*d&9{AluTp;3S8ODeVPr*MgqRl+WQ7UeADvrJ-l{#=~A7!!Pc2?tTfdxYfaL;'
    '2%vAkX1J&BX~kL#E32JHmG9z7m8)zMW_2eS?LANHSf;0n>hTULR)ZEz^I}ihbHhs-W0vOymdpjYd1(gZ#BJfJGn4`DB$N+$U6u0`6vi`Ck(QY<'
    '_aE`ZtF>I7&s;YAS~SJ5d^t$<1F|hg&`}H~p=&u$w2Tv#;*&-hmag`iFLPxPIp5jobT8ctUFT$@r{}6ods(LyYU>=UjgjXp(`tkRwKaBVm|fNc'
    'F<3^Q4%$5j@I{}D+Gvy~##|F=86i+6gn7Cg!BjEs+IbcjmS3%EbHjGkn$ZYiP+5(tY`e^dhili1m5C|bRnnQVw4$q#I8R+>19JU<dGQfv+S=t='
    'wVr-bO##K|3_C#gF{$?!g}&AzP34wYn)?`-<HtE*<|!22J6^9rGrmuY*XcAF@X8C<S+WCQkmhPf9w5+O{*0O!V)-3W#s0biaH}|C<G!7z2NCNV'
    '7XVdV+p`FuRtrO<mx{loC>p%|hYhjCX};BxiSB(m&x?%je!^du4JoL{;X{zqp-))Zu)|sH+dIx5Gm^Dba{RiX1-bKYSHZV)@>Q!4ff~)ZW_`VY'
    'ikp93TAK~ZqX1NBam-YGKZjZ%>$x;0OCIoCM7Aj}&#GcLU;;}dI^Y6^KJw`l`8Rx!?)*CW`F+nZ5>_KcHcjN|2gjjqjOxfIQ9lO5ln(;9N=b89'
    '-kl>QmiwO7iY6-Jq#}}EzSqvoO8qIfEqWG!a;r=apc+)Az3#T?2+xA@H__d0wdju_W3;MuwT~b5AerHGG;(-zJSTz}f>1$sWda?>qfu~&exjbq'
    'DH!#@v<G;OigKgOY}K_uuI5^;w7qn;2F*Qim1{_nrfn5+#Dh4O;rdLp$yY7PORZ=wD=C=JH{P9h)-UI+{p-vA%1K-*wwex-YS%pEqX|~-fOKK}'
    'L9uFkNurw&w{z<5D`)K{nVIrY?sypE;qNSpN`KV68v+_UXi}UzdnKt;lh5)ZVmEqLC!Yl4&s8OlQ*YJPRewyM=<<nyH1D(+{y^Mu<{I-|zD6Z;'
    'ZdA^A2Oerdxo<CU&UD-OP`!4kDPe5kSkBbnuv8sgyytD6xG{@o(If)#)ol#iS{@X|&@7JjXe-p)qSLiAb<dffv;pCsAPwZV3S%M;>kS<%ur~+l'
    ')r?M^Ti<Oiwm$3ZY<+TUc-Fa_x=rC^l8@C#dS^jwHHLGXLFon299I^wE0Nz)3fyZBrpUu_#PM)NDz0Q1j^Vfgd_Fl5`YgWc&5Pprq<8IeQvgOS'
    'IzbzeuXvU(O^Ia!TWX|$)sXe-GsOw&8&fVUWQ>cV*3Mi~GyygOTx5*29?cXMDgdWGgfi@<sa<H+ijzr)CAQ0j!!%6E25DED9gh@yrs#9GU6+A#'
    '@oWI{D9QMg2L7<TbXjBVhSpq>z187p>F`hu)<qbek+Y8wE!tqy_hs1q!WSacwE9vVDBGE1Vmeolc4wA02%TA}oI_N*p~MWpa4!O@V{4xrnH}^)'
    '^gCkSzR|@>!-zw@CSEAdr0YX#nre6+38c~~9)nkENom_ed@8GbE_0rnPPb-H8)@7X&M^hBhEaxdUwWdhr|}5%a)PmhX+Ip5@H*SC#C~}fcJzQZ'
    'BKrzuJX=DiMkaXp9M6*!At+>kGn$FF>B+9jKa#ULh2p<b$y__9uFi~o$1t(bWO_Y{`@-ci11+3SCzDYeVW=){Oc%HxRorIj^yXGvT`3>c706Mo'
    '-}CNrUF2`nk~Z+Z;6L31>8$9pO9_wJF&b>DrG*}39OprJ7seypR`=ZQ%9Ck=8;L6a{;t^ju)nLs3<gw}p`xr+>9e$kO4z>PS*#3oN}(~=OqWSE'
    'O#h-I4MdQn{o##|{+)k4?`#c^A+(}r`&bqiXN=CVCI`?hmm3&qt$?8dqgiU2etLu2ZqtxM1i+un#4yhCVqT2S|H)i%w2bD(^dpm&JtYksXVCo<'
    '8f#1d_h&S=`31Ah{gw0Wubgjx<$U{B$oV#ZL$BIE%)%rG;XdZ<Mr!l7db_^A)Q)0RVPR4@v&mX+INZnY`V|+H&mXPlJL}oaAzWbH0=RAgT(<zO'
    '&jV<shO=_VEtzr7nNUt6O*7|OrpOXm=By2w#VP0X;dHITY}PSYx}g*EsxQy4%V*7ca<VCBlCamK^!^>X9S-otew_DVDn!i}Pgw?W?Sd(La;RWH'
    '%*$t=7Dph%VX39VYk5SZ3NWmaqIH~5?MC~fG=~pCJb0+UW;f$F@whQ(Ql>wiK#;C4jtb*2Ct00*pj>)<U;deS2ESCgKkIbpm&o?_1=67DlggP+'
    'OfT?%At{<6WftcY(~JMJbBq0z^6Y<O$}>G<(j!xz)fDNnna`Z+yGnp&7;6jipw-GYWF^xl&QjUQ{(Ul_eX+b?3zdkLB}=F0ve~`Nv1-in^>vvz'
    '%c!98X8pM-#;V4p-fwJlxjbRk1pA|@!Z@?m%{a{EsB4EbIHA|3IvQ$b7|^+a>N`$>(0lEusZ-fEojg@=0zCn4lNtEbMvUaT%L3Dp#3Z_L0wY6L'
    'jh}litO0ZjZ#>5&K63KH+{Ia5;3O?eURor@{t^k#^=8m~qVqqx_`JW3N0hO^qZ%qb)Au}vb_Ng~xIazu-=<OYkI3tE9kJ6`ruqEgs0^*1AN-`V'
    'iy3+&0XR;x-bx42eLNtSyH&KazGFn&nCawt!ZYgEJxq&=xzo}0D5H{oGPW9hE!t7rv2C~7wKa(30dr<y=xPINHnIJ=;)$los7>$M#4hQXXgRUU'
    'm-S8c<#%$II6^4*&Gnp6RxY5?qVoXNAyQS;K$n(6cr#D6ep_oix~3%`D1o`Izizo8c-1+0yV1pG;m!R4b;7hdn_cn&?h)|IyqW3EtaQfLGA*6!'
    'fM{bZr#V?n@Y=Vh0dJaOW;4!Gk87}*V8(NuE1SbDy3ItYo}LP@i7?Lhk8NbRL%HSRTsO=fsXkltkJD<9Egzp5xV-R;?c<NpDr9?2Cvy$H=$WZf'
    'o!&(%rzcCTo!gzrf$O<5tv>b`Ze0#lVn`~1XByG}2?tgG`DabFBL6O@N&h-e+HjuquM?$zohbd6JyE*UnNnC@uM{Dei`i-squ}%>vy~me0bMJn'
    'Q8?~|_h{YmS3H?E5Z5z3hO7#7gZ^YQL6K^H6%&TLE+PnqQ_K+^1R@?|gqxtGIH9x+WvPBT8gX<<?uXZXRdg>LQ3Mp0lds6!GM~(JvY7AiHHYs}'
    'KpH66tjIm>KnMB_FvSf0#Jq^3a6(U%P#6>5QT`Y?n_+Teaq)exEUOW?7Jt1$g~wCOoQla;K*wen(445+rB5j?vj~3QV~lD8SL#lV=@48Fbf$tO'
    'KVlD6rgF}s2J1cvyu)>#<C2BJv?>I+>N$uR8FXINFg<8c$`2=#C>eN0DG5q38Wq%t&l{eeMsIbNM~};MrF?Em?tBlc$n?InyKw_6;CIwYTYv=_'
    'r(lXmOGM1c90sdgSSba#vDS?`8)jcThN`DimTy!><(B+KpQ}zI8iPQlZIQMjO+PCGHh+=jc1m79H%~+Xvrb_#cWx#LHj1coWhIq?Ixn`>4YfLh'
    '<F?=y-Z{(ET?E*OZatQHXfVQ+1QmsyMh`^?g^C!ZH!(2bl+x+QHYXU?$Sgp)*+)@s#J6%RU^*e2Ag2)7Q5J=RSw~%?I}j&tUdb41Y>(y@kSvd~'
    'yGUSz`6R?EXw}f=>4+BkzBs}_S^X415~&DRkQ^w?qJD_yZBe2PC=3^a+hkCs9JWNJu7R<c3GCV^>afef41hH<VkR=#{U&<2>v<%*_G_Me8}r4='
    'flOR4LWu`ahL@edn1D2A$oBX`+#lQG=T;Rs8np3$wv$Rjf{V?hv}}uqpp+^~f~wj)&Eycb+U0kGYJZkYj+uV>sO&x6>Gj}AL8gGNwc7e?32UvA'
    'R0IMkM?f6OyIxZsag|ehO|}Lw_}48q0Q?4O5|*HxkIiuJJNr&2e6^Eutd;ZWxD-I>C(S)cb<^c{X}zH2m{J@`zm*}BOkY-M2)SG9e)-m9bjm-c'
    'F_3f#F0OSIp@6EoF6SP3qEp&*RU@`zE>p<;0G}V~pP3g?BuM5*>Auq@fIIZ>bIHF1WJjo5J>Z>{x)5wCf+TG!1yzZCdakG4RJCL}d%6*$dIaPv'
    'U{w)00*n|b-Nut(R=hdilEFrzln;Y6;t*##M#aiXv*hrc0j_uaT6XP{^&dks@w`{|85AfDx_#=FS^Tk3HL5$|N@jj6SP4lQR+-`xnuMzgs5UvJ'
    '+Ur$Q)hbn*;1ZJ^CGmDh#$C2`I1OdMTL1k213aICH2'
)


def unpack_tadisr():
    payload = zlib.decompress(base64.b85decode(UPSTREAM_B85))
    if hashlib.sha256(payload).hexdigest() != UPSTREAM_SHA256:
        raise ValueError('Sai SHA-256 của mã TADiSR nhúng')
    folder = RUNTIME / UPSTREAM_REVISION
    for name, source in json.loads(payload).items():
        path = bundle_file(folder, name)
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(source, encoding='utf8')
    return folder

In [ ]:
WORKER_SOURCE = r'''
import json
import sys
import time
from pathlib import Path


def starts(length, tile, overlap):
    if not 0 <= overlap < tile:
        raise ValueError('Require 0 <= overlap < tile')
    if length <= tile:
        return [0]
    values = list(range(0, length - tile + 1, tile - overlap))
    if values[-1] != length - tile:
        values.append(length - tile)
    return values


def tiled_tadisr(image, predict, tile, overlap, *, device):
    """Blend on the accumulation device, with independent starts on each axis."""
    import torch
    from torch.nn import functional as F
    if tile % 32 or overlap % 32:
        raise ValueError('TADiSR tile/overlap must be multiples of 32')
    h, w = image.shape[-2:]
    image = F.pad(image, (0, max(0, tile - w), 0, max(0, tile - h)), mode='replicate')
    ph, pw = image.shape[-2:]
    sums = torch.zeros(1, 3, ph, pw, device=device)
    masks = torch.zeros(1, 1, ph, pw, device=device)
    weights = torch.zeros(1, 1, ph, pw, device=device)
    blend = torch.hann_window(tile, periodic=False, device=device).clamp_min(0.001)
    blend = (blend[:, None] * blend[None, :])[None, None]
    ys, xs = (starts(ph, tile, overlap), starts(pw, tile, overlap))
    seam_diffs = []
    for top in ys:
        for left in xs:
            sr, mask = predict(image[:, :, top:top + tile, left:left + tile])
            sr = sr.to(device=device, dtype=torch.float32)
            mask = mask.to(device=device, dtype=torch.float32)
            if tuple(sr.shape) != (1, 3, tile, tile) or tuple(mask.shape) != (1, 1, tile, tile):
                raise ValueError('TADiSR output violates tile contract')
            old_weights = weights[:, :, top:top + tile, left:left + tile]
            old = sums[:, :, top:top + tile, left:left + tile]
            if top or left:
                occupied = old_weights > 0
                previous = old / old_weights.clamp_min(1e-12)
                diff = (previous - sr).abs().masked_fill(~occupied, 0).sum()
                seam_diffs.append(diff / (3 * occupied.sum()).clamp_min(1))
            old += sr * blend
            masks[:, :, top:top + tile, left:left + tile] += mask * blend
            old_weights += blend
    if not (weights > 0).all():
        raise ValueError('Tiled output has uncovered pixels')
    seam_mae = torch.stack(seam_diffs).max().item() if seam_diffs else 0.0
    return ((sums / weights)[:, :, :h, :w], (masks / weights)[:, :, :h, :w],
            {'tiles': len(ys) * len(xs), 'overlap_patch_mae': seam_mae,
             'covered_all_pixels': True, 'visual_seam_review': 'required'})


def restoration_memory_limits(torch, options):
    cap = options.get('max_memory', '18GiB')
    if not isinstance(cap, str) or not cap.endswith('GiB'):
        raise ValueError('max_memory must be a per-GPU limit such as 18GiB')
    requested = int(float(cap[:-3]) * 1024**3)
    reserve = int(float(options.get('activation_reserve_gib', 4)) * 1024**3)
    if requested <= 0 or reserve < 1024**3:
        raise ValueError('Positive weight budget and >=1 GiB activation reserve required')
    limits = {}
    for index in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(index)
        limit = min(requested, free - reserve)
        # VAE, decoder and prompt live on the entry GPU outside the sharded model.
        if index == 0:
            limit -= int(options.get('entry_reserve_gib', 2) * 1024**3)
        if limit < 4 * 1024**3:
            raise MemoryError(f'GPU {index}: insufficient free VRAM after activation reserve')
        limits[index] = limit
    return limits


def verify_restoration_device_map(mapping, count, *, require_multiple=True):
    devices = set()
    for value in mapping.values():
        if isinstance(value, int):
            index = value
        else:
            label = str(value)
            if not label.startswith('cuda:'):
                raise MemoryError(f'Model would spill to {label}; reduce scope or change explicit placement budgets')
            index = int(label.split(':')[1])
        if not 0 <= index < count:
            raise ValueError(f'Device map refers to unavailable GPU {index}')
        devices.add(index)
    if require_multiple and len(devices) < 2:
        raise ValueError('Sharded placement did not distribute the model across multiple GPUs')
    return mapping


def load_tadisr_multi_gpu(assets, options):
    import pathlib
    import torch
    from accelerate import dispatch_model, infer_auto_device_map
    from tadisr.inference import CogView4Config, load_cogview4
    # Construct/load on host RAM; never materialize the complete float32 stack on L4 0.
    print('Nạp VAE, transformer và adapter CogView4 trên CPU.', flush=True)
    net = load_cogview4(CogView4Config(pathlib.Path(assets['base_model']),
        pathlib.Path(assets['checkpoint']), pathlib.Path(assets['prompt_embeddings']), 'cpu'))
    limits = restoration_memory_limits(torch, options)
    no_split = getattr(net.transformer, '_no_split_modules', None) or ['CogView4Block']
    mapping = infer_auto_device_map(net.transformer, max_memory=limits,
        no_split_module_classes=no_split, dtype=torch.float32)
    verify_restoration_device_map(mapping, len(limits))
    print(f'Chuyển transformer lên GPU: {len(set(mapping.values()))} GPU.', flush=True)
    net.transformer = dispatch_model(net.transformer, device_map=mapping, main_device=0, force_hooks=True)
    net.execution_device = torch.device('cuda:0')
    print('Chuyển VAE và text decoder lên GPU đầu.', flush=True)
    net.vae.to(net.execution_device)
    net.js_decoder.to(net.execution_device)
    net.prompt_embeds = net.prompt_embeds.to(net.execution_device)
    net._vtm_device_map = {k: str(v) for k, v in mapping.items()}
    return net


def restoration_cuda_metrics(torch, *, reset=False, devices=None):
    records = []
    if torch.cuda.is_available():
        for index in devices if devices is not None else range(torch.cuda.device_count()):
            torch.cuda.synchronize(index)
            if reset:
                torch.cuda.reset_peak_memory_stats(index)
            records.append(dict(index=index, peak_allocated_bytes=torch.cuda.max_memory_allocated(index),
                                peak_reserved_bytes=torch.cuda.max_memory_reserved(index)))
    return records


def probe_gpus(torch, options):
    if not torch.cuda.is_available() or torch.cuda.device_count() < 2:
        raise RuntimeError('TADiSR chế độ sharded cần ít nhất hai GPU CUDA.')
    limits = restoration_memory_limits(torch, options)
    if sum(limits.values()) < 28 * 1024**3:
        raise MemoryError('Tổng VRAM dành cho weights dưới 28 GiB. Kiểm tra số GPU và VRAM trống.')
    for index, limit in limits.items():
        probe = torch.ones(8, device=f'cuda:{index}').sum().item()
        if probe != 8:
            raise RuntimeError(f'CUDA probe thất bại trên GPU {index}')
        print(f'GPU {index}: {torch.cuda.get_device_name(index)}; '
              f'ngân sách weights {limit / 1024**3:.1f} GiB')


def load_tadisr_bf16(torch, assets):
    from contextlib import ExitStack
    from unittest.mock import patch
    from tadisr.pipelines import AutoencoderKL, CogView4Pipeline, CogView4Transformer2DModel

    device = torch.device('cuda:0')

    def load_on_gpu(load, label):
        def from_pretrained(path, **kwargs):
            print(f'Nạp {label} BF16 trực tiếp lên {device}.', flush=True)
            return load(path, torch_dtype=torch.bfloat16, device_map={'': device},
                        low_cpu_mem_usage=True, local_files_only=True, **kwargs)
        return from_pretrained

    with ExitStack() as stack:
        for component, label in ((AutoencoderKL, 'VAE'), (CogView4Transformer2DModel, 'transformer')):
            stack.enter_context(patch.object(
                component, 'from_pretrained', load_on_gpu(component.from_pretrained, label)))
        net = CogView4Pipeline(ckpt_dir=assets['base_model'],
                               prompt_path=assets['prompt_embeddings'], device=str(device))
    # SHA-256 đã được kiểm tra ở cell chính; đọc adapter một lần bằng loader upstream.
    print('Nạp LoRA, VAE skip weights và text decoder.', flush=True)
    net.load_model(assets['checkpoint'])
    net.set_eval()
    net.to(dtype=torch.bfloat16)
    net._vtm_device_map = {'': str(device)}
    return net


def probe_single_gpu(torch, weight_bytes):
    if not torch.cuda.is_available() or torch.cuda.get_device_capability(0)[0] < 8:
        raise RuntimeError('BF16 cần GPU CUDA Ampere trở lên. Có thể chọn fp32_sharded trên nhiều GPU.')
    gib = 1024 ** 3
    free, _ = torch.cuda.mem_get_info(0)
    if free < weight_bytes + 4 * gib:
        raise MemoryError(f'{free / gib:.1f} GiB VRAM trống; cần khoảng '
                          f'{weight_bytes / gib + 4:.1f} GiB cho weights và vùng làm việc. '
                          'Chọn GPU lớn hơn hoặc fp32_sharded.')
    result = torch.ones((16, 16), dtype=torch.bfloat16, device='cuda:0')
    if (result @ result)[0, 0].item() != 16:
        raise RuntimeError('CUDA BF16 probe thất bại.')
    print(f'GPU 0: {torch.cuda.get_device_name(0)}; {free / gib:.1f} GiB VRAM trống.')


def inference_tiles(torch, model, pixels, job, bf16):
    from torch.nn import functional as F

    height, width = pixels.shape[-2] * 4, pixels.shape[-1] * 4
    overlap = job['overlap']
    tiles = [job['tile']] + [size for size in (320, 256, 192, 128)
                             if overlap < size < job['tile']]
    # Keep a smoke page and its blend buffers on GPU. Use host buffers for large pages.
    fast_blend = bf16 and height * width <= 4_000_000
    device = model.execution_device if fast_blend else 'cpu'
    attempts = []

    def attempt(tile, accumulation_device):
        torch.manual_seed(job['seed'])
        completed = 0
        total = len(starts(height, tile, overlap)) * len(starts(width, tile, overlap))
        print(f'Tile {tile} px: {total} lượt, ghép trên {accumulation_device}.')

        def predict(patch):
            nonlocal completed
            encoder = model.vae.encoder
            try:
                result = model(patch.to(model.execution_device))
            finally:
                # The upstream encoder keeps a reference to the current tile's skip features.
                if hasattr(encoder, 'current_down_blocks'):
                    encoder.current_down_blocks = []
            completed += 1
            if completed == 1 or completed % 10 == 0 or completed == total:
                print(f'Tile {completed}/{total}')
            return result

        try:
            # Keep the source FP32 for the scheduler lookup; autocast GPU operations.
            upsampled = F.interpolate(pixels.to(accumulation_device), size=(height, width),
                                      mode='bicubic', align_corners=False)
            with torch.inference_mode(), torch.autocast('cuda', enabled=bf16, dtype=torch.bfloat16):
                return tiled_tadisr(upsampled, predict, tile, overlap, device=accumulation_device)
        except torch.cuda.OutOfMemoryError:
            return None

    for tile in tiles:
        attempts.append(tile)
        result = attempt(tile, device)
        if result is not None:
            sr, mask, diagnostics = result
            diagnostics.update(tile=tile, tile_attempts=attempts, accumulation_device=str(device))
            return sr, mask, diagnostics
        # The failed frame has unwound, so cached temporary allocations can be released.
        torch.cuda.empty_cache()
        print(f'CUDA OOM với tile {tile} px. Giữ model trong VRAM và thử tile nhỏ hơn nếu còn.')

    raise RuntimeError(f'TADiSR hết VRAM sau các tile {attempts}.')


def prepare_prompt(torch, job):
    import runpy
    from unittest.mock import patch
    from transformers import GlmModel

    mode = job['prompt_device']
    if mode == 'auto':
        can_use_gpu = (torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
                       and torch.cuda.mem_get_info(0)[0] >= job['encoder_bytes'] + 3 * 1024 ** 3)
        mode = 'cuda:0' if can_use_gpu else 'cpu'
    print(f'Nạp text encoder BF16 trên {mode}.', flush=True)
    glm_load = GlmModel.from_pretrained

    def load_encoder(path):
        options = {'dtype': torch.bfloat16, 'local_files_only': True}
        if mode != 'cpu':
            options['device_map'] = {'': torch.device(mode)}
        encoder = glm_load(path, **options)
        encoder.config.use_cache = False
        return encoder

    script = Path(job['vendor_dir']) / 'scripts/prepare_prompt_embeddings.py'
    sys.argv = [str(script), '--base-model', job['assets']['base_model'],
                '--output', job['prompt_output'], '--device', mode]
    with patch.object(GlmModel, 'from_pretrained', load_encoder), torch.inference_mode():
        runpy.run_path(str(script), run_name='__main__')


def prepare_input(job):
    from PIL import Image, ImageOps

    path = Path(job['source'])
    if path.suffix.lower() == '.pdf':
        import pypdfium2 as pdfium

        document = pdfium.PdfDocument(path)
        try:
            number = job['pdf_page']
            if not 1 <= number <= len(document):
                raise ValueError(f'PDF_PAGE={number}, PDF chỉ có {len(document)} trang')
            page = document[number - 1]
            try:
                rendered = page.render(scale=job['pdf_dpi'] / 72)
                try:
                    image = rendered.to_pil().convert('RGB')
                finally:
                    rendered.close()
            finally:
                page.close()
        finally:
            document.close()
    else:
        with Image.open(path) as source:
            image = ImageOps.exif_transpose(source).convert('RGB')
    source_size = image.size
    edge = job['max_input_edge']
    if edge is not None:
        image.thumbnail((edge, edge), Image.Resampling.LANCZOS)
    image.save(Path(job['output']) / 'original.png')
    return image, source_size


def stage_checkpoints(job):
    import hashlib
    import shutil

    root = Path(job['model_dir'])
    stage = Path(job['stage_dir'])
    stage.mkdir(parents=True, exist_ok=True)
    pending = []
    for record in job['inference_files']:
        relative = record['path']
        target = stage / relative
        if target.is_file() and target.stat().st_size == record['bytes']:
            continue
        pending.append(record)
    gib = 1024 ** 3
    needed = sum(record['bytes'] for record in pending)
    if needed and shutil.disk_usage(stage).free < needed + gib:
        print(f'Ổ tạm thiếu chỗ cho {needed / gib:.2f} GiB weights; đọc trực tiếp từ Dataset.')
        return
    print(f'Checkpoint cục bộ: cần chép thêm {needed / gib:.2f} GiB.')
    for record in pending:
        relative = record['path']
        target = stage / relative
        target.parent.mkdir(parents=True, exist_ok=True)
        partial = target.with_suffix(target.suffix + f'.{Path(job["output"]).name}.part')
        digest = hashlib.sha256()
        copied = 0
        next_report = 0
        print(f'Đang đọc {relative} ({record["bytes"] / gib:.2f} GiB).')
        try:
            with (root / relative).open('rb') as source, partial.open('wb') as output:
                for chunk in iter(lambda: source.read(16 * 1024 ** 2), b''):
                    output.write(chunk)
                    digest.update(chunk)
                    copied += len(chunk)
                    if time.monotonic() >= next_report:
                        print(f'  {copied / gib:.2f}/{record["bytes"] / gib:.2f} GiB')
                        next_report = time.monotonic() + 15
            if copied != record['bytes'] or digest.hexdigest() != record['sha256']:
                raise ValueError(f'Sai checksum của bản sao: {relative}')
            partial.replace(target)
            print(f'Đã chép và kiểm SHA-256: {relative}')
        finally:
            partial.unlink(missing_ok=True)
    (Path(job['output']) / 'checkpoint_staged.ok').touch()


def main():
    job = json.loads(Path(sys.argv[1]).read_text(encoding='utf8'))
    action = sys.argv[2]
    if action == 'stage':
        import signal

        def stop_copy(_signal, _frame):
            raise SystemExit(143)

        signal.signal(signal.SIGTERM, stop_copy)
        stage_checkpoints(job)
        return

    if action == 'input':
        image, source_size = prepare_input(job)
        output = Path(job['output'])
        (output / 'input.json').write_text(json.dumps({
            'source_size': source_size, 'input_size': image.size,
        }), encoding='utf8')
        print(f'Đầu vào: {source_size} → {image.size}')
        return
    import torch

    bf16 = job['precision'] == 'bf16'
    if bf16:
        probe_single_gpu(torch, sum(record['bytes'] for record in job['inference_files']))
    else:
        probe_gpus(torch, job['options'])
    if action == 'prompt':
        prepare_prompt(torch, job)
        return

    import numpy as np
    from PIL import Image
    from tadisr.inference import save_mask, save_rgb

    options = job['options']
    output = Path(job['output'])
    torch.manual_seed(job['seed'])
    print(f'Đang nạp CogView4 với cấu hình {job["precision"]}.')
    started = time.monotonic()
    devices = [0] if bf16 else None
    restoration_cuda_metrics(torch, reset=True, devices=devices)
    if bf16:
        model = load_tadisr_bf16(torch, job['assets'])
    else:
        model = load_tadisr_multi_gpu(job['assets'], options)
    metrics_after_load = restoration_cuda_metrics(torch, reset=True, devices=devices)
    load_seconds = time.monotonic() - started
    print(f'Đã nạp model sau {load_seconds:.1f} giây.')
    with Image.open(output / 'original.png') as image:
        width, height = image.size
        pixels = np.asarray(image, dtype=np.float32) / 255
    tensor = torch.from_numpy(pixels).permute(2, 0, 1)[None]
    started = time.monotonic()
    sr, mask, diagnostics = inference_tiles(torch, model, tensor, job, bf16)
    metrics_after_inference = restoration_cuda_metrics(torch, devices=devices)
    inference_seconds = time.monotonic() - started
    if not torch.isfinite(sr).all() or not torch.isfinite(mask).all():
        raise ValueError('TADiSR trả về pixel hoặc mask không hữu hạn')
    save_rgb(sr, output / 'tadisr_x4.png')
    save_mask(mask, output / 'text_mask.png')
    inputs = json.loads((output / 'input.json').read_text(encoding='utf8'))
    pipeline_started = job.pop('pipeline_started')
    report = {**job, **inputs, 'output_size': [width * 4, height * 4],
              'pipeline_seconds': time.monotonic() - pipeline_started,
              'load_seconds': load_seconds, 'inference_seconds': inference_seconds,
              'diagnostics': diagnostics, 'device_map': model._vtm_device_map,
              'transformer_dtype': str(model.transformer.dtype), 'vae_dtype': str(model.vae.dtype),
              'cuda_after_load': metrics_after_load,
              'cuda_after_inference': metrics_after_inference}
    temporary = output / 'run.json.part'
    temporary.write_text(json.dumps(report, indent=2), encoding='utf8')
    temporary.replace(output / 'run.json')


if __name__ == '__main__':
    main()
'''

In [ ]:
def log_message(log, line):
    print(line, end='', flush=True)
    log.write(line)
    log.flush()


def process_activity(pid):
    proc = Path('/proc') / str(pid)
    try:
        fields = (proc / 'stat').read_text().rsplit(')', 1)[-1].split()
        cpu = (int(fields[11]) + int(fields[12])) / os.sysconf('SC_CLK_TCK')
        rss = int(fields[21]) * os.sysconf('SC_PAGE_SIZE') / 1024 ** 3
        return f'PID {pid}, state={fields[0]}, CPU tích lũy {cpu:.1f}s, RAM {rss:.2f} GiB'
    except (OSError, ValueError, IndexError):
        return f'PID {pid}'


def run_logged(command, env, log, *, stage, timeout_minutes=None, ready_prefix=None):
    lines = queue.Queue()
    start = time.monotonic()
    deadline = start + timeout_minutes * 60 if timeout_minutes is not None else None
    heartbeat = start + 30
    process = subprocess.Popen(list(map(str, command)), env=env, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, encoding='utf8',
                               errors='replace', bufsize=1)

    def read_output():
        try:
            with process.stdout as stdout:
                for line in stdout:
                    lines.put(line)
        finally:
            lines.put(None)

    threading.Thread(target=read_output, daemon=True).start()
    output_done = False
    try:
        while not output_done or process.poll() is None:
            now = time.monotonic()
            try:
                line = lines.get(timeout=1)
            except queue.Empty:
                if now >= heartbeat:
                    log_message(log, f'{stage}: {now - start:.0f}s, {process_activity(process.pid)}.\n')
                    heartbeat = now + 30
            else:
                if line is None:
                    output_done = True
                else:
                    log_message(log, line)
                    heartbeat = now + 30
                    if ready_prefix is not None and line.startswith(ready_prefix):
                        deadline = None
                        stage = 'Inference'
            if deadline is not None and now >= deadline and process.poll() is None:
                message = f'{stage}: quá {timeout_minutes:g} phút. {process_activity(process.pid)}.'
                log_message(log, message + '\n')
                raise TimeoutError(message)
        if process.returncode:
            raise subprocess.CalledProcessError(process.returncode, command)
    finally:
        if process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                process.kill()


if not INPUT_ROOT.is_dir() or not OUTPUT_ROOT.parent.is_dir() or sys.platform != 'linux':
    raise RuntimeError('Mở notebook trên Kaggle với GPU và Internet Off.')
if TILE < 32 or TILE % 32 or OVERLAP % 32 or not 0 <= OVERLAP < TILE:
    raise ValueError('TILE và OVERLAP phải là bội của 32; 0 <= OVERLAP < TILE.')
if PDF_PAGE < 1 or PDF_DPI <= 0 or (MAX_INPUT_EDGE is not None and MAX_INPUT_EDGE < 1):
    raise ValueError('Kiểm tra PDF_PAGE, PDF_DPI và MAX_INPUT_EDGE.')
if PRECISION not in ('bf16', 'fp32_sharded'):
    raise ValueError('PRECISION phải là bf16 hoặc fp32_sharded.')
if PROMPT_DEVICE not in ('auto', 'cpu'):
    raise ValueError('PROMPT_DEVICE phải là auto hoặc cpu.')
min_gpus = 1 if PRECISION == 'bf16' else 2
if GPU_IDS is not None and (len(set(GPU_IDS)) != len(GPU_IDS)
                            or len(GPU_IDS) < min_gpus or any(i < 0 for i in GPU_IDS)):
    raise ValueError(f'GPU_IDS cần ít nhất {min_gpus} ID khác nhau, hoặc None.')

if min(STAGE_TIMEOUT_MINUTES, PROMPT_TIMEOUT_MINUTES, LOAD_TIMEOUT_MINUTES) <= 0:
    raise ValueError('Các giới hạn thời gian phải lớn hơn 0.')

pipeline_started = time.monotonic()
packages = find_bundle(PACKAGES_DIR, PLAN_PATH, 'packages', 'PACKAGES_DIR')
models = find_bundle(MODELS_DIR, ASSET_PATH, 'models', 'MODELS_DIR')
source = find_input(INPUT_FILE)
is_pdf = source.suffix.lower() == '.pdf'
model_dir = (models / ASSET_PATH).parent
prompt_override = input_path(PROMPT_FILE) if PROMPT_FILE else None
if prompt_override is not None and not prompt_override.is_file():
    raise FileNotFoundError(prompt_override)
print('Đang kiểm tra manifest và SHA-256 của adapter TADiSR.')
base, adapter, staged_prompt, inference_files, encoder_bytes = verify_assets(model_dir, prompt_override)
print(f'Packages: {packages}\nModels: {models}\nNguồn: {source}')

env = prep_worker_environment()
env.update(USE_TF='0', HF_ENABLE_PARALLEL_LOADING=str(PARALLEL_SHARD_LOADING).lower(),
           HF_PARALLEL_LOADING_WORKERS='2')
if GPU_IDS is not None:
    env['CUDA_VISIBLE_DEVICES'] = ','.join(map(str, GPU_IDS))
python = install_overlay(packages, is_pdf, env)
vendor_dir = unpack_tadisr()
env['PYTHONPATH'] = str(vendor_dir)
worker_path = RUNTIME / 'worker.py'
worker_path.write_text(WORKER_SOURCE, encoding='utf8')

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
run_dir = Path(tempfile.mkdtemp(prefix='run_', dir=OUTPUT_ROOT))
print(f'Kết quả: {run_dir}')
prompt_key = hashlib.sha256(
    (UPSTREAM_SHA256 + sha256(model_dir / 'assets.json')).encode()
).hexdigest()[:20]
prompt_name = 'saved_prompt_tokens_nocfg.pt'
legacy_prompt = RUNTIME / 'prompts' / prompt_key / prompt_name
bf16_prompt = RUNTIME / 'prompts' / f'{prompt_key}_bf16' / prompt_name
prompt = staged_prompt or (legacy_prompt if legacy_prompt.is_file() else bf16_prompt)
job = {
    'source': str(source), 'pdf_page': PDF_PAGE if is_pdf else None,
    'pdf_dpi': PDF_DPI if is_pdf else None, 'max_input_edge': MAX_INPUT_EDGE,
    'output': str(run_dir), 'tile': TILE, 'overlap': OVERLAP, 'seed': SEED,
    'upstream_revision': UPSTREAM_REVISION, 'pipeline_started': pipeline_started,
    'parallel_shard_loading': PARALLEL_SHARD_LOADING,
    'precision': PRECISION, 'encoder_bytes': encoder_bytes, 'prompt_device': PROMPT_DEVICE,
    'vendor_dir': str(vendor_dir),
    'model_dir': str(model_dir), 'inference_files': inference_files,
    'stage_dir': str(Path('/tmp/tadisr_smoke') / prompt_key),
    'assets': {'base_model': str(base), 'checkpoint': str(adapter),
               'prompt_embeddings': str(prompt)},
    'options': {'placement': 'sharded', 'max_memory': MAX_GPU_MEMORY,
                'activation_reserve_gib': 4, 'entry_reserve_gib': 6},
}
job_file = run_dir / 'job.json'
job_file.write_text(json.dumps(job, indent=2), encoding='utf8')
worker_command = [python, '-u', worker_path, job_file]
with (run_dir / 'run.log').open('w', encoding='utf8') as log:
    run_logged([*worker_command, 'input'], env, log, stage='Đọc ảnh')
    job['prompt_cache_hit'] = prompt.is_file()
    started = time.monotonic()
    if not job['prompt_cache_hit']:
        log_message(log, 'Đang tạo prompt embedding offline ở BF16.\n')
        prompt.parent.mkdir(parents=True, exist_ok=True)
        with tempfile.TemporaryDirectory(dir=prompt.parent) as temporary:
            derived = Path(temporary) / prompt.name
            prompt_job = Path(temporary) / 'prompt_job.json'
            prompt_job.write_text(json.dumps({**job, 'prompt_output': str(derived)}), encoding='utf8')
            run_logged([python, '-u', worker_path, prompt_job, 'prompt'], env, log,
                       stage='Tạo prompt', timeout_minutes=PROMPT_TIMEOUT_MINUTES)
            derived.replace(prompt)
    else:
        log_message(log, f'Dùng lại prompt embedding: {prompt}\n')
    job['prompt_seconds'] = time.monotonic() - started
    exported_prompt = OUTPUT_ROOT / 'prompts' / prompt_key / prompt.name
    exported_prompt.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(dir=exported_prompt.parent) as temporary:
        export_tmp = Path(temporary) / exported_prompt.name
        shutil.copyfile(prompt, export_tmp)
        export_tmp.replace(exported_prompt)
    log_message(log, f'Prompt để dùng lại ở phiên sau: {exported_prompt}\n')

    started = time.monotonic()
    if STAGE_WEIGHTS:
        run_logged([*worker_command, 'stage'], env, log, stage='Chép checkpoint',
                   timeout_minutes=STAGE_TIMEOUT_MINUTES)
        if (run_dir / 'checkpoint_staged.ok').is_file():
            local = Path(job['stage_dir'])
            job['assets'].update(base_model=str(local / 'CogView4'),
                                 checkpoint=str(local / ADAPTER_NAME))
    job['staging_seconds'] = time.monotonic() - started
    job_file.write_text(json.dumps(job, indent=2), encoding='utf8')
    run_logged([*worker_command, 'inference'], env, log, stage='Nạp model',
               timeout_minutes=LOAD_TIMEOUT_MINUTES, ready_prefix='Đã nạp model sau ')

In [ ]:
from IPython.display import display
from PIL import Image

result = read_json(run_dir / 'run.json')
print(f"{result['input_size']} → {result['output_size']}; "
      f"{result['diagnostics']['tiles']} tile × {result['diagnostics']['tile']} px; "
      f"inference {result['inference_seconds']:.1f} giây; "
      f"tổng {result['pipeline_seconds']:.1f} giây")
for filename in ('original.png', 'tadisr_x4.png', 'text_mask.png'):
    print(filename)
    with Image.open(run_dir / filename) as preview:
        preview.thumbnail((900, 900))
        display(preview.copy())

In [ ]:
# Tải ảnh của lượt chạy vừa hoàn tất.
from zipfile import ZipFile

from IPython.display import FileLink, display

if 'run_dir' not in globals() or not (run_dir / 'run.json').is_file():
    raise RuntimeError('Hãy chạy xong cell TADiSR trước khi tải ảnh.')

images = [run_dir / name for name in ('original.png', 'tadisr_x4.png', 'text_mask.png')]
missing = [image.name for image in images if not image.is_file()]
if missing:
    raise FileNotFoundError(f'Thiếu ảnh trong {run_dir}: {missing}')

archive = run_dir / 'tadisr_images.zip'
with ZipFile(archive, 'w') as bundle:
    for image in images:
        bundle.write(image, arcname=image.name)
display(FileLink(str(archive)))